# CRP, body temperature, and pediatric appendicitis

This notebook examines whether body temperature modifies the association between C-reactive protein (CRP) and appendicitis diagnosis in the Regensburg Pediatric Appendicitis Dataset (version 1.01).

## Analysis overview

1. Inspect the source data, define the diagnosis outcome, and document excluded variables and quality-control decisions.
2. Describe missing data and patient characteristics; compare functional forms for CRP and temperature.
3. Estimate the CRP–temperature interaction with modified Poisson regression and adjusted logistic regression.
4. Assess risk-scale interpretation using marginal standardization; examine sensitivity to missingness, transformations, flexible functional forms, and ultrasound information.
5. Evaluate diagnostic performance, out-of-sample prediction, and the broader family of 253 pairwise predictor interactions.

The primary analyses are **associational, not causal**. The same dataset was used for exploratory modeling and follow-up sensitivity analyses. In particular, ultrasound was kept out of the main predictor screen because imaging findings helped establish some patients' diagnoses; imaging-based checks are reported separately.

**Input:** `app_data.xlsx` in the notebook's directory (see README). **Outputs:** generated tables, figures, and model summaries under `interaction_outputs_final_clean/`. The source workbook is not included with the notebook.


## 1. Reproducibility setup

In [1]:
from pathlib import Path
from itertools import combinations
import json, platform, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy
from scipy.stats import chi2, pearsonr, spearmanr
from statsmodels.nonparametric.smoothers_lowess import lowess

import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

import patsy
import sklearn
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix,
)

RANDOM_STATE = 42
TARGET = "Diagnosis_binary"
DATA_PATH = Path("app_data.xlsx")

OUT = Path("interaction_outputs_final_clean")
TAB = OUT / "tables"
FIG = OUT / "figures"
MODEL_OUT = OUT / "model_outputs"
for p in (OUT, TAB, FIG, MODEL_OUT):
    p.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)

versions = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
    "patsy": patsy.__version__,
    "scikit_learn": sklearn.__version__,
}
print(json.dumps(versions, indent=2))
(OUT / "software_versions.json").write_text(json.dumps(versions, indent=2))

C:\Users\Asus\anaconda4\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


{
  "python": "3.11.4",
  "platform": "Windows-10-10.0.26200-SP0",
  "numpy": "1.26.4",
  "pandas": "2.3.2",
  "scipy": "1.11.4",
  "statsmodels": "0.14.0",
  "patsy": "1.0.2",
  "scikit_learn": "1.8.0"
}


204

## 2. Load raw data and verify the source dataset

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Place app_data.xlsx in the same directory as this notebook."
    )

df_raw = pd.read_excel(DATA_PATH)
print("Raw shape:", df_raw.shape)
print("Diagnosis counts including missing:")
display(df_raw["Diagnosis"].value_counts(dropna=False).rename("n").to_frame())

if "Diagnosis" not in df_raw.columns:
    raise KeyError("Required column 'Diagnosis' was not found.")
if df_raw.shape[0] != 782:
    raise ValueError(f"Expected 782 source records, found {df_raw.shape[0]}. Check dataset version.")

Raw shape: (782, 58)
Diagnosis counts including missing:


,n
Diagnosis,
appendicitis,463
no appendicitis,317
NaN,2


## 3. Define the binary diagnosis outcome

In [3]:
diagnosis_map = {
    "appendicitis": 1,
    "no appendicitis": 0,
}

df = df_raw.copy()
raw_diag = df["Diagnosis"].astype("string").str.strip().str.lower()
unexpected = sorted(set(raw_diag.dropna().unique()) - set(diagnosis_map))
if unexpected:
    raise ValueError(f"Unexpected Diagnosis labels: {unexpected}")

df[TARGET] = raw_diag.map(diagnosis_map)
df = df.loc[df[TARGET].notna()].copy()
df[TARGET] = df[TARGET].astype(int)

print("Outcome-defined cohort n:", len(df))
print(df[TARGET].value_counts().sort_index().rename(index={0:"No appendicitis",1:"Appendicitis"}))
print("Appendicitis prevalence: {:.1%}".format(df[TARGET].mean()))

assert len(df) == 780
assert int(df[TARGET].sum()) == 463
assert int((df[TARGET] == 0).sum()) == 317

Outcome-defined cohort n: 780
Diagnosis_binary
No appendicitis    317
Appendicitis       463
Name: count, dtype: int64
Appendicitis prevalence: 59.4%


## 4. Define predictor eligibility and exclusions

The candidate set is defined **before modeling**. This prevents later results from determining which variables are allowed into the exploratory predictor family.

Ultrasound-derived variables are removed here and do not appear in primary, adjusted, exploratory-interaction, or prediction models. Post-diagnostic variables, composite diagnostic scores, workflow/technical fields, free-text fields, highly incomplete complication variables, and redundant variables are also excluded from the candidate predictor set.


In [4]:
POST_DIAGNOSTIC_OR_OUTCOME_VARS = [
    "Length_of_Stay", "Management", "Severity"
]
COMPOSITE_OR_DIAGNOSIS_PROXY_VARS = [
    "Diagnosis_Presumptive", "Alvarado_Score", "Paedriatic_Appendicitis_Score"
]
TECHNICAL_OR_WORKFLOW_VARS = ["US_Performed", "US_Number"]
FREE_TEXT_VARS = [
    "Lymph_Nodes_Location", "Abscess_Location", "Gynecological_Findings"
]
REDUNDANT_VARS = ["Height", "Weight", "Neutrophilia"]
ULTRASOUND_VARS = ["Appendix_Diameter", "Appendix_on_US", "Free_Fluids"]
HIGH_MISSINGNESS_OR_COMPLICATION_VARS = [
    "Appendix_Wall_Layers", "Target_Sign", "Surrounding_Tissue_Reaction",
    "Pathological_Lymph_Nodes", "Bowel_Wall_Thickening", "Ileus",
    "Coprostasis", "Meteorism", "Enteritis", "Conglomerate_of_Bowel_Loops",
    "Appendicolith", "Perforation", "Appendicular_Abscess", "Perfusion",
    "Segmented_Neutrophils"
]

CONTINUOUS_PREDICTORS = [
    "Age", "BMI", "Body_Temperature", "WBC_Count", "Neutrophil_Percentage",
    "Hemoglobin", "Thrombocyte_Count", "CRP"
]
BINARY_PREDICTORS = [
    "Sex", "Migratory_Pain", "Lower_Right_Abd_Pain",
    "Contralateral_Rebound_Tenderness", "Ipsilateral_Rebound_Tenderness",
    "Coughing_Pain", "Psoas_Sign", "Nausea", "Loss_of_Appetite", "Dysuria"
]
CATEGORICAL_PREDICTORS = [
    "Peritonitis", "Stool", "Ketones_in_Urine", "RBC_in_Urine", "WBC_in_Urine"
]
CANDIDATE_PREDICTORS = CONTINUOUS_PREDICTORS + BINARY_PREDICTORS + CATEGORICAL_PREDICTORS

assert len(CANDIDATE_PREDICTORS) == 23
assert len(set(CANDIDATE_PREDICTORS)) == 23
assert not (set(ULTRASOUND_VARS) & set(CANDIDATE_PREDICTORS))
missing_cols = [v for v in CANDIDATE_PREDICTORS if v not in df.columns]
if missing_cols:
    raise KeyError(f"Candidate variables missing from dataset: {missing_cols}")

print("Candidate predictor count:", len(CANDIDATE_PREDICTORS))
print("Planned pairwise interactions:", len(list(combinations(CANDIDATE_PREDICTORS, 2))))
display(pd.DataFrame({
    "type": ["Continuous", "Binary", "Categorical"],
    "n": [len(CONTINUOUS_PREDICTORS), len(BINARY_PREDICTORS), len(CATEGORICAL_PREDICTORS)]
}))

Candidate predictor count: 23
Planned pairwise interactions: 253


,type,n
0,Continuous,8
1,Binary,10
2,Categorical,5


## 5. Source-value quality control and recoding

In [5]:
# Keep an audit of source values recoded to missing rather than guessing corrections.
qc_recodes = []

def set_missing_if_equal(frame, col, value, reason):
    mask = pd.to_numeric(frame[col], errors="coerce").eq(value)
    n = int(mask.sum())
    if n:
        frame.loc[mask, col] = np.nan
    qc_recodes.append({"variable": col, "value": value, "n_recoded": n, "reason": reason})

set_missing_if_equal(df, "Body_Temperature", 26.9, "clinically implausible isolated value")
set_missing_if_equal(df, "Hemoglobin", 36.0, "implausible measurement-scale value")
# Source-QC-only variables; retained in df but not candidate predictors.
set_missing_if_equal(df, "RBC_Count", 14.0, "implausible measurement-scale value")
for x in (79.2, 84.6, 86.9):
    set_missing_if_equal(df, "RDW", x, "implausible measurement-scale value")

mixed_stool = df["Stool"].astype("string").str.strip().eq("constipation, diarrhea")
qc_recodes.append({
    "variable":"Stool", "value":"constipation, diarrhea",
    "n_recoded":int(mixed_stool.sum()), "reason":"non-standard mixed category"
})
df.loc[mixed_stool, "Stool"] = np.nan

# Coerce all retained continuous predictors to numeric.
for v in CONTINUOUS_PREDICTORS:
    original_nonmissing = df[v].notna()
    converted = pd.to_numeric(df[v], errors="coerce")
    failures = int((original_nonmissing & converted.isna()).sum())
    if failures:
        raise ValueError(f"{failures} non-missing values in {v} failed numeric conversion")
    df[v] = converted

display(pd.DataFrame(qc_recodes))
pd.DataFrame(qc_recodes).to_csv(TAB / "source_value_qc_recodes.csv", index=False)

,variable,value,n_recoded,reason
0,Body_Temperature,26.9,1,clinically implausible isolated value
1,Hemoglobin,36.0,1,implausible measurement-scale value
2,RBC_Count,14.0,1,implausible measurement-scale value
3,RDW,79.2,1,implausible measurement-scale value
4,RDW,84.6,1,implausible measurement-scale value
5,RDW,86.9,1,implausible measurement-scale value
6,Stool,"constipation, diarrhea",1,non-standard mixed category


## 6. CRP transformation and standardization

In [6]:
if (df["CRP"].dropna() < 0).any():
    raise ValueError("Negative CRP values detected")

df["log_CRP1"] = np.log1p(df["CRP"])

crp_skew_raw = df["CRP"].skew()
crp_skew_log = df["log_CRP1"].skew()
print(f"Raw CRP skewness: {crp_skew_raw:.3f}")
print(f"log(CRP+1) skewness: {crp_skew_log:.3f}")

# Full outcome-defined cohort scaling for inferential models.
scaling = {}
for v in ["log_CRP1", "Body_Temperature"]:
    mu = df[v].mean(skipna=True)
    sd = df[v].std(skipna=True, ddof=1)
    scaling[v] = {"mean": float(mu), "sd": float(sd)}
    df[f"z_{v}"] = (df[v] - mu) / sd

df["z_Body_Temperature_sq"] = df["z_Body_Temperature"] ** 2

print("Scaling constants:")
display(pd.DataFrame(scaling).T)
pd.DataFrame(scaling).T.to_csv(TAB / "scaling_constants.csv")

Raw CRP skewness: 2.932
log(CRP+1) skewness: 0.253
Scaling constants:


,mean,sd
log_CRP1,2.165375,1.700179
Body_Temperature,37.418088,0.821436


## 7. Missingness among selected predictors

In [7]:
missingness = pd.DataFrame({
    "variable": CANDIDATE_PREDICTORS,
    "nonmissing_n": [int(df[v].notna().sum()) for v in CANDIDATE_PREDICTORS],
    "missing_n": [int(df[v].isna().sum()) for v in CANDIDATE_PREDICTORS],
    "missing_pct": [100 * df[v].isna().mean() for v in CANDIDATE_PREDICTORS],
}).sort_values("missing_pct", ascending=False).reset_index(drop=True)

display(missingness)
missingness.to_csv(TAB / "candidate_predictor_missingness_23.csv", index=False)

complete_all_23 = df[CANDIDATE_PREDICTORS + [TARGET]].dropna().shape[0]
print(f"Complete cases across all 23 candidate predictors: {complete_all_23}/{len(df)} "
      f"({100*complete_all_23/len(df):.1f}%)")

,variable,nonmissing_n,missing_n,missing_pct
0,RBC_in_Urine,576,204,26.153846
1,Ketones_in_Urine,582,198,25.384615
2,WBC_in_Urine,583,197,25.256410
3,Ipsilateral_Rebound_Tenderness,619,161,20.641026
4,Neutrophil_Percentage,679,101,12.948718
5,Psoas_Sign,745,35,4.487179
6,Dysuria,753,27,3.461538
7,BMI,754,26,3.333333
8,Hemoglobin,763,17,2.179487
9,Thrombocyte_Count,764,16,2.051282


Complete cases across all 23 candidate predictors: 403/780 (51.7%)


## 8. Descriptive statistics by diagnosis

In [8]:
def summarize_continuous(frame, var, use_median=False):
    rows = []
    for y, label in [(0,"No appendicitis"),(1,"Appendicitis")]:
        x = frame.loc[frame[TARGET].eq(y), var].dropna()
        if use_median:
            val = f"{x.median():.2f} [{x.quantile(.25):.2f}, {x.quantile(.75):.2f}]"
        else:
            val = f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"
        rows.append({"variable":var, "group":label, "value":val, "n_nonmissing":len(x)})
    return rows

desc_rows = []
for v in CONTINUOUS_PREDICTORS:
    desc_rows += summarize_continuous(df, v, use_median=(v=="CRP"))

for v in BINARY_PREDICTORS + CATEGORICAL_PREDICTORS:
    for y, label in [(0,"No appendicitis"),(1,"Appendicitis")]:
        s = df.loc[df[TARGET].eq(y), v]
        denom = int(s.notna().sum())
        for level, n in s.value_counts(dropna=True).items():
            desc_rows.append({
                "variable":v, "group":label, "level":level,
                "n":int(n), "denominator":denom,
                "percent":100*n/denom if denom else np.nan
            })

descriptive_long = pd.DataFrame(desc_rows)
descriptive_long.to_csv(TAB / "descriptive_statistics_long.csv", index=False)
print("Saved descriptive_statistics_long.csv")

Saved descriptive_statistics_long.csv


## 9. Functional-form assessment

LOWESS is used only as an exploratory shape diagnostic. Formal model choice compares linear, quadratic, and 4-df natural cubic spline logistic specifications on the same observations.

In [9]:
# 9A. Body temperature
from patsy import cr

temp_df = df[[TARGET, "Body_Temperature"]].dropna().copy()
temp_mean = temp_df["Body_Temperature"].mean()
temp_df["temp_c"] = temp_df["Body_Temperature"] - temp_mean
temp_df["temp_c2"] = temp_df["temp_c"] ** 2

temp_linear = smf.glm(f"{TARGET} ~ temp_c", data=temp_df, family=sm.families.Binomial()).fit()
temp_quadratic = smf.glm(f"{TARGET} ~ temp_c + temp_c2", data=temp_df, family=sm.families.Binomial()).fit()
temp_spline = smf.glm(f"{TARGET} ~ cr(Body_Temperature, df=4)", data=temp_df, family=sm.families.Binomial()).fit()

lr_temp = 2 * (temp_quadratic.llf - temp_linear.llf)
p_lr_temp = chi2.sf(lr_temp, 1)

temp_form = pd.DataFrame([
    {"specification":"Linear", "n":len(temp_df), "log_likelihood":temp_linear.llf, "AIC":temp_linear.aic, "BIC":temp_linear.bic_llf},
    {"specification":"Quadratic", "n":len(temp_df), "log_likelihood":temp_quadratic.llf, "AIC":temp_quadratic.aic, "BIC":temp_quadratic.bic_llf},
    {"specification":"Natural cubic spline (4 df)", "n":len(temp_df), "log_likelihood":temp_spline.llf, "AIC":temp_spline.aic, "BIC":temp_spline.bic_llf},
])
display(temp_form)
print(f"Linear vs quadratic LR chi-square={lr_temp:.3f}, p={p_lr_temp:.6f}")
temp_form.to_csv(TAB / "temperature_functional_form.csv", index=False)

# LOWESS plot
plot_temp = temp_df.sort_values("Body_Temperature")
sm_temp = lowess(plot_temp[TARGET], plot_temp["Body_Temperature"], frac=.35, it=0, return_sorted=True)
rng = np.random.default_rng(RANDOM_STATE)
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(plot_temp["Body_Temperature"], plot_temp[TARGET] + rng.normal(0,.015,len(plot_temp)), alpha=.25, s=18)
ax.plot(sm_temp[:,0], sm_temp[:,1], linewidth=2)
ax.set(xlabel="Body temperature (°C)", ylabel="Appendicitis diagnosis (jittered)", title="Body temperature and appendicitis diagnosis")
fig.tight_layout(); fig.savefig(FIG / "temperature_lowess.png", dpi=300); plt.close(fig)

,specification,n,log_likelihood,AIC,BIC
0,Linear,774,-513.901467,1031.802935,1041.106079
1,Quadratic,774,-511.240886,1028.481772,1042.436488
2,Natural cubic spline (4 df),774,-510.461810,1028.923621,1047.529908


Linear vs quadratic LR chi-square=5.321, p=0.021068


In [10]:
# 9B. log(CRP+1)
crp_df = df[[TARGET, "log_CRP1"]].dropna().copy()
crp_mean = crp_df["log_CRP1"].mean()
crp_df["log_CRP_c"] = crp_df["log_CRP1"] - crp_mean
crp_df["log_CRP_c2"] = crp_df["log_CRP_c"] ** 2

crp_linear = smf.glm(f"{TARGET} ~ log_CRP_c", data=crp_df, family=sm.families.Binomial()).fit()
crp_quadratic = smf.glm(f"{TARGET} ~ log_CRP_c + log_CRP_c2", data=crp_df, family=sm.families.Binomial()).fit()
crp_spline = smf.glm(f"{TARGET} ~ cr(log_CRP1, df=4)", data=crp_df, family=sm.families.Binomial()).fit()

lr_crp = 2 * (crp_quadratic.llf - crp_linear.llf)
p_lr_crp = chi2.sf(lr_crp, 1)
crp_form = pd.DataFrame([
    {"specification":"Linear", "n":len(crp_df), "log_likelihood":crp_linear.llf, "AIC":crp_linear.aic, "BIC":crp_linear.bic_llf},
    {"specification":"Quadratic", "n":len(crp_df), "log_likelihood":crp_quadratic.llf, "AIC":crp_quadratic.aic, "BIC":crp_quadratic.bic_llf},
    {"specification":"Natural cubic spline (4 df)", "n":len(crp_df), "log_likelihood":crp_spline.llf, "AIC":crp_spline.aic, "BIC":crp_spline.bic_llf},
])
display(crp_form)
print(f"Linear vs quadratic LR chi-square={lr_crp:.3f}, p={p_lr_crp:.6f}")
crp_form.to_csv(TAB / "crp_functional_form.csv", index=False)

plot_crp = crp_df.sort_values("log_CRP1")
sm_crp = lowess(plot_crp[TARGET], plot_crp["log_CRP1"], frac=.35, it=0, return_sorted=True)
rng = np.random.default_rng(RANDOM_STATE)
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(plot_crp["log_CRP1"], plot_crp[TARGET] + rng.normal(0,.015,len(plot_crp)), alpha=.25, s=18)
ax.plot(sm_crp[:,0], sm_crp[:,1], linewidth=2)
ax.set(xlabel="log(CRP + 1)", ylabel="Appendicitis diagnosis (jittered)", title="Transformed CRP and appendicitis diagnosis")
fig.tight_layout(); fig.savefig(FIG / "crp_lowess.png", dpi=300); plt.close(fig)

,specification,n,log_likelihood,AIC,BIC
0,Linear,771,-458.067727,920.135454,929.430830
1,Quadratic,771,-457.737285,921.474571,935.417636
2,Natural cubic spline (4 df),771,-452.768764,913.537529,932.128282


Linear vs quadratic LR chi-square=0.661, p=0.416248


## 10. CRP–temperature relationship, correlations, and VIF diagnostics

In [11]:
# Scientifically relevant bivariate relationship.
ct = df[["Body_Temperature", "log_CRP1"]].dropna().copy().sort_values("Body_Temperature")
pearson_r, pearson_p = pearsonr(ct["Body_Temperature"], ct["log_CRP1"])
spearman_r, spearman_p = spearmanr(ct["Body_Temperature"], ct["log_CRP1"])
print(f"CRP-temperature Pearson r={pearson_r:.3f}, p={pearson_p:.3g}")
print(f"CRP-temperature Spearman rho={spearman_r:.3f}, p={spearman_p:.3g}; n={len(ct)}")

ct_sm = lowess(ct["log_CRP1"], ct["Body_Temperature"], frac=.35, it=0, return_sorted=True)
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(ct["Body_Temperature"], ct["log_CRP1"], alpha=.25, s=18)
ax.plot(ct_sm[:,0], ct_sm[:,1], linewidth=2)
ax.set(xlabel="Body temperature (°C)", ylabel="log(CRP + 1)", title="CRP and body temperature")
fig.tight_layout(); fig.savefig(FIG / "crp_temperature_scatter_lowess.png", dpi=300); plt.close(fig)

# Numerical correlation matrix only; no publication heatmap.
corr_vars = ["Age", "BMI", "Body_Temperature", "WBC_Count", "Neutrophil_Percentage", "Hemoglobin", "Thrombocyte_Count", "log_CRP1"]
corr_matrix = df[corr_vars].corr(method="pearson")
corr_matrix.to_csv(TAB / "continuous_correlation_matrix.csv")
display(corr_matrix.round(3))

# VIF with quadratic temperature term.
vif_vars = ["Age", "BMI", "z_Body_Temperature", "z_Body_Temperature_sq", "WBC_Count", "Neutrophil_Percentage", "Hemoglobin", "Thrombocyte_Count", "log_CRP1"]
vif_df = df[vif_vars].dropna().copy()
Xv = sm.add_constant(vif_df, has_constant="add")
vif_table = pd.DataFrame({
    "variable": [c for c in Xv.columns if c != "const"],
    "VIF": [variance_inflation_factor(Xv.values, i) for i,c in enumerate(Xv.columns) if c != "const"]
}).sort_values("VIF", ascending=False)
print("VIF complete-case n:", len(vif_df))
display(vif_table)
vif_table.to_csv(TAB / "vif_table.csv", index=False)

CRP-temperature Pearson r=0.472, p=1.02e-43
CRP-temperature Spearman rho=0.473, p=6.81e-44; n=766


,Age,BMI,Body_Temperature,WBC_Count,Neutrophil_Percentage,Hemoglobin,Thrombocyte_Count,log_CRP1
Age,1.000,0.480,-0.292,-0.248,-0.178,0.346,-0.262,-0.203
BMI,0.480,1.000,-0.172,-0.123,-0.184,0.176,-0.035,-0.070
Body_Temperature,-0.292,-0.172,1.000,0.280,0.341,-0.103,-0.005,0.472
WBC_Count,-0.248,-0.123,0.280,1.000,0.663,-0.088,0.317,0.432
Neutrophil_Percentage,-0.178,-0.184,0.341,0.663,1.000,-0.078,0.093,0.467
Hemoglobin,0.346,0.176,-0.103,-0.088,-0.078,1.000,-0.130,-0.125
Thrombocyte_Count,-0.262,-0.035,-0.005,0.317,0.093,-0.130,1.000,0.075
log_CRP1,-0.203,-0.070,0.472,0.432,0.467,-0.125,0.075,1.000


VIF complete-case n: 649


,variable,VIF
4,WBC_Count,2.059430
5,Neutrophil_Percentage,2.050324
2,z_Body_Temperature,1.726300
0,Age,1.659252
8,log_CRP1,1.568145
1,BMI,1.343887
3,z_Body_Temperature_sq,1.286772
7,Thrombocyte_Count,1.232734
6,Hemoglobin,1.168663


## 11. Modified Poisson helper and primary CRP × temperature model

In [12]:
def fit_modified_poisson(formula, data):
    return smf.glm(formula=formula, data=data, family=sm.families.Poisson()).fit(cov_type="HC0")

def exponentiated_table(res, label="model"):
    ci = res.conf_int()
    rows=[]
    for term in res.params.index:
        if term == "Intercept":
            continue
        rows.append({
            "model": label, "term": term,
            "estimate_exp": np.exp(res.params[term]),
            "CI_lower": np.exp(ci.loc[term,0]),
            "CI_upper": np.exp(ci.loc[term,1]),
            "p_value": res.pvalues[term], "n": int(res.nobs)
        })
    return pd.DataFrame(rows)

PRIMARY_FORMULA = (
    f"{TARGET} ~ z_log_CRP1 + z_Body_Temperature + z_Body_Temperature_sq + "
    "z_log_CRP1:z_Body_Temperature"
)
primary_data = df[[TARGET,"z_log_CRP1","z_Body_Temperature","z_Body_Temperature_sq"]].dropna().copy()
poisson_crp_temperature = fit_modified_poisson(PRIMARY_FORMULA, primary_data)
primary_table = exponentiated_table(poisson_crp_temperature, "Primary CRP x temperature")
display(primary_table)
primary_table.to_csv(TAB / "primary_modified_poisson.csv", index=False)

term = "z_log_CRP1:z_Body_Temperature"
print("Primary interaction:", primary_table.loc[primary_table.term.eq(term)].to_dict("records")[0])

,model,term,estimate_exp,CI_lower,CI_upper,p_value,n
0,Primary CRP x temperature,z_log_CRP1,1.390831,1.313220,1.473029,2.046141e-29,766
1,Primary CRP x temperature,z_Body_Temperature,1.035741,0.954809,1.123532,3.975750e-01,766
2,Primary CRP x temperature,z_Body_Temperature_sq,0.974682,0.931583,1.019775,2.664288e-01,766
3,Primary CRP x temperature,z_log_CRP1:z_Body_Temperature,0.922840,0.865814,0.983621,1.360901e-02,766


Primary interaction: {'model': 'Primary CRP x temperature', 'term': 'z_log_CRP1:z_Body_Temperature', 'estimate_exp': 0.9228397867097494, 'CI_lower': 0.865814492435478, 'CI_upper': 0.9836209481073811, 'p_value': 0.013609005437977717, 'n': 766}


## 12. Multivariable modified Poisson model

In [13]:
ADJUSTMENT_VARS = [
    "Age", "BMI", "Sex", "WBC_Count", "Neutrophil_Percentage",
    "Migratory_Pain", "Lower_Right_Abd_Pain", "Contralateral_Rebound_Tenderness",
    "Coughing_Pain", "Psoas_Sign", "Nausea", "Loss_of_Appetite", "Dysuria", "Peritonitis"
]

ADJUSTMENT_TERMS = [
    "Age", "BMI", "C(Sex)", "WBC_Count", "Neutrophil_Percentage",
    "C(Migratory_Pain)", "C(Lower_Right_Abd_Pain)", "C(Contralateral_Rebound_Tenderness)",
    "C(Coughing_Pain)", "C(Psoas_Sign)", "C(Nausea)", "C(Loss_of_Appetite)",
    "C(Dysuria)", "C(Peritonitis, Treatment(reference='no'))"
]

adjusted_required = [TARGET, "z_log_CRP1", "z_Body_Temperature", "z_Body_Temperature_sq"] + ADJUSTMENT_VARS
adjusted_data = df[adjusted_required].dropna().copy()
ADJUSTED_FORMULA = PRIMARY_FORMULA + " + " + " + ".join(ADJUSTMENT_TERMS)
poisson_crp_temperature_adjusted = fit_modified_poisson(ADJUSTED_FORMULA, adjusted_data)
adjusted_table = exponentiated_table(poisson_crp_temperature_adjusted, "Adjusted CRP x temperature")
display(adjusted_table)
adjusted_table.to_csv(TAB / "adjusted_modified_poisson.csv", index=False)
print("Adjusted n:", int(poisson_crp_temperature_adjusted.nobs))
print("Adjusted interaction:", adjusted_table.loc[adjusted_table.term.eq(term)].to_dict("records")[0])

,model,term,estimate_exp,CI_lower,CI_upper,p_value,n
0,Adjusted CRP x temperature,C(Sex)[T.male],1.132490,0.999343,1.283377,5.121669e-02,606
1,Adjusted CRP x temperature,C(Migratory_Pain)[T.yes],1.140928,1.000744,1.300749,4.871517e-02,606
2,Adjusted CRP x temperature,C(Lower_Right_Abd_Pain)[T.yes],1.477046,0.979889,2.226441,6.247195e-02,606
3,Adjusted CRP x temperature,C(Contralateral_Rebound_Tenderness)[T.yes],1.106706,0.972826,1.259010,1.232737e-01,606
4,Adjusted CRP x temperature,C(Coughing_Pain)[T.yes],1.138739,0.999737,1.297068,5.046420e-02,606
5,Adjusted CRP x temperature,C(Psoas_Sign)[T.yes],0.900619,0.781761,1.037548,1.471915e-01,606
6,Adjusted CRP x temperature,C(Nausea)[T.yes],1.078102,0.927404,1.253288,3.276230e-01,606
7,Adjusted CRP x temperature,C(Loss_of_Appetite)[T.yes],1.038252,0.901085,1.196299,6.035911e-01,606
8,Adjusted CRP x temperature,C(Dysuria)[T.yes],0.968886,0.736229,1.275067,8.215156e-01,606
9,Adjusted CRP x temperature,"C(Peritonitis, Treatment(reference='no'))[T.ge...",1.267525,1.050323,1.529643,1.343944e-02,606


Adjusted n: 606
Adjusted interaction: {'model': 'Adjusted CRP x temperature', 'term': 'z_log_CRP1:z_Body_Temperature', 'estimate_exp': 0.9194627974057689, 'CI_lower': 0.8539524468879552, 'CI_upper': 0.989998727556976, 'p_value': 0.02598066135303384, 'n': 606}


## 13. Nonlinear interaction sensitivity: CRP × T and CRP × T²

In [14]:
NONLINEAR_INT_FORMULA = ADJUSTED_FORMULA + " + z_log_CRP1:z_Body_Temperature_sq"
poisson_interaction_nonlinear = fit_modified_poisson(NONLINEAR_INT_FORMULA, adjusted_data)
nonlinear_table = exponentiated_table(poisson_interaction_nonlinear, "Adjusted nonlinear interaction")
display(nonlinear_table[nonlinear_table.term.str.contains("z_log_CRP1")])

interaction_terms = ["z_log_CRP1:z_Body_Temperature", "z_log_CRP1:z_Body_Temperature_sq"]
R = np.zeros((2, len(poisson_interaction_nonlinear.params)))
for i,t in enumerate(interaction_terms):
    R[i, poisson_interaction_nonlinear.params.index.get_loc(t)] = 1
wald = poisson_interaction_nonlinear.wald_test(R, scalar=True)
print(f"Joint robust Wald test for both interaction terms: chi2={float(wald.statistic):.3f}, p={float(wald.pvalue):.6f}")
nonlinear_table.to_csv(TAB / "nonlinear_interaction_sensitivity.csv", index=False)

,model,term,estimate_exp,CI_lower,CI_upper,p_value,n
11,Adjusted nonlinear interaction,z_log_CRP1,1.193046,1.094107,1.300932,0.000064,606
14,Adjusted nonlinear interaction,z_log_CRP1:z_Body_Temperature,0.897901,0.825941,0.976131,0.011511,606
19,Adjusted nonlinear interaction,z_log_CRP1:z_Body_Temperature_sq,1.050799,0.990613,1.114642,0.099651,606


Joint robust Wald test for both interaction terms: chi2=7.129, p=0.028310


## 14. Conditional CRP risk ratios across body temperature

In [15]:
res = poisson_crp_temperature_adjusted
b_crp = res.params["z_log_CRP1"]
b_int = res.params[term]
cov = res.cov_params()
muT = scaling["Body_Temperature"]["mean"]
sdT = scaling["Body_Temperature"]["sd"]

def conditional_crp_rr(temp_c):
    zt = (temp_c - muT) / sdT
    logrr = b_crp + zt*b_int
    var = (cov.loc["z_log_CRP1","z_log_CRP1"] + zt**2*cov.loc[term,term]
           + 2*zt*cov.loc["z_log_CRP1",term])
    se = np.sqrt(var)
    return np.exp(logrr), np.exp(logrr-1.96*se), np.exp(logrr+1.96*se)

selected_temps = [36.5, 37.5, 38.5]
cond_rows=[]
for t in selected_temps:
    rr,lo,hi = conditional_crp_rr(t)
    cond_rows.append({"temperature_C":t,"RR_per_1SD_log_CRP":rr,"CI_lower":lo,"CI_upper":hi})
conditional_selected = pd.DataFrame(cond_rows)
display(conditional_selected)
conditional_selected.to_csv(TAB / "conditional_crp_rr_selected_temperatures.csv", index=False)

# Plot over the observed temperature range of the adjusted sample.
temps = np.linspace(adjusted_data["z_Body_Temperature"].min()*sdT+muT,
                    adjusted_data["z_Body_Temperature"].max()*sdT+muT, 300)
curve=[]
for t in temps:
    rr,lo,hi = conditional_crp_rr(t)
    curve.append((t,rr,lo,hi))
curve = pd.DataFrame(curve, columns=["temperature_C","RR","CI_lower","CI_upper"])
curve.to_csv(TAB / "conditional_crp_rr_curve.csv", index=False)

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(7,7), gridspec_kw={"height_ratios":[2.2,1]})
ax1.plot(curve.temperature_C, curve.RR, linewidth=2)
ax1.fill_between(curve.temperature_C, curve.CI_lower, curve.CI_upper, alpha=.2)
ax1.axhline(1, linestyle="--", linewidth=1)
ax1.set(ylabel="RR per 1-SD higher log(CRP + 1)", title="Conditional CRP association across body temperature")

# Same analysis sample distribution.
temp_actual = df.loc[adjusted_data.index, "Body_Temperature"]
ax2.hist(temp_actual.dropna(), bins=20, edgecolor="black")
ax2.set(xlabel="Body temperature (°C)", ylabel="Count")
fig.tight_layout(); fig.savefig(FIG / "conditional_crp_rr_with_temperature_distribution.png", dpi=300); plt.close(fig)

,temperature_C,RR_per_1SD_log_CRP,CI_lower,CI_upper
0,36.5,1.364244,1.221492,1.523679
1,37.5,1.231684,1.143792,1.326330
2,38.5,1.112004,0.984065,1.256577


## 15. Logistic-regression effect-scale sensitivity

In [16]:
def fit_robust_logistic(formula, data):
    return smf.glm(formula=formula, data=data, family=sm.families.Binomial()).fit(cov_type="HC0")

logit_primary = fit_robust_logistic(PRIMARY_FORMULA, primary_data)
logit_adjusted = fit_robust_logistic(ADJUSTED_FORMULA, adjusted_data)

logit_rows=[]
for label,res in [("CRP and temperature terms",logit_primary),("Multivariable adjusted",logit_adjusted)]:
    ci=res.conf_int().loc[term]
    logit_rows.append({"analysis":label,"n":int(res.nobs),"interaction_OR":np.exp(res.params[term]),
                       "CI_lower":np.exp(ci.iloc[0]),"CI_upper":np.exp(ci.iloc[1]),"p_value":res.pvalues[term]})
logistic_sensitivity = pd.DataFrame(logit_rows)
display(logistic_sensitivity)
logistic_sensitivity.to_csv(TAB / "logistic_effect_scale_sensitivity.csv", index=False)

,analysis,n,interaction_OR,CI_lower,CI_upper,p_value
0,CRP and temperature terms,766,0.844867,0.693157,1.029781,0.095046
1,Multivariable adjusted,606,0.821954,0.635107,1.063772,0.136191


## 16. Exploratory 253-pair interaction screen

All pairwise combinations of the specified 23 candidate predictors are evaluated. No univariable-significance prefilter is used. Temperature is represented by linear and quadratic standardized terms, so any interaction involving temperature is tested jointly across the corresponding interaction columns. Sparse or non-estimable categorical interactions are retained in the planned FDR family but conservatively assigned p=1 for multiplicity correction.

In [17]:
# Model-space helpers for the complete 23-predictor family.
SCREEN_CONTINUOUS = ["Age","BMI","Body_Temperature","WBC_Count","Neutrophil_Percentage","Hemoglobin","Thrombocyte_Count","CRP"]
SCREEN_CATEGORICAL = [v for v in CANDIDATE_PREDICTORS if v not in SCREEN_CONTINUOUS]
ALL_PAIRS = list(combinations(CANDIDATE_PREDICTORS, 2))
assert len(ALL_PAIRS) == 253

# Work from cleaned df; use log-CRP whenever the source variable CRP is modeled.
def standardize_series(s):
    s = pd.to_numeric(s, errors="coerce")
    sd=s.std(ddof=1)
    return (s-s.mean())/sd if pd.notna(sd) and sd>0 else pd.Series(np.nan,index=s.index)

def variable_block(data, var):
    if var == "Body_Temperature":
        z = standardize_series(data[var])
        return pd.DataFrame({"Body_Temperature_linear":z, "Body_Temperature_quadratic":z**2}, index=data.index)
    if var == "CRP":
        z = standardize_series(np.log1p(pd.to_numeric(data[var],errors="coerce")))
        return pd.DataFrame({"CRP_log_standardized":z}, index=data.index)
    if var in SCREEN_CONTINUOUS:
        z=standardize_series(data[var])
        return pd.DataFrame({var:z}, index=data.index)
    d = pd.get_dummies(data[var].astype("category"), prefix=var, drop_first=True, dtype=float)
    return d

def sparse_check(data, v1, v2):
    # Only categorical structure can create sparse level cells in this diagnostic.
    cats=[v for v in (v1,v2) if v in SCREEN_CATEGORICAL]
    if not cats:
        return False
    if len(cats)==1:
        tab=pd.crosstab(data[cats[0]], data[TARGET], dropna=False)
    else:
        joint=data[cats].astype(str).agg(" | ".join,axis=1)
        tab=pd.crosstab(joint, data[TARGET], dropna=False)
    if tab.empty:
        return True
    # Ensure both outcome columns exist.
    for y in [0,1]:
        if y not in tab.columns: tab[y]=0
    totals=tab[[0,1]].sum(axis=1)
    return bool((totals<10).any() or (tab[[0,1]]<5).any(axis=None))

def fit_pair_screen(data, v1, v2, adjustment_vars=None):
    required=[TARGET,v1,v2]
    if adjustment_vars:
        required += [v for v in adjustment_vars if v not in (v1,v2)]
    required=list(dict.fromkeys(required))
    tmp=data.dropna(subset=required).copy()
    sparse=sparse_check(tmp,v1,v2)
    if len(tmp)<30 or tmp[TARGET].nunique()<2:
        return {"var1":v1,"var2":v2,"n":len(tmp),"p_value":np.nan,"sparse_flag":sparse,"reliable":False,"reason":"insufficient sample/outcome variation"}
    try:
        A=variable_block(tmp,v1); B=variable_block(tmp,v2)
        if A.shape[1]==0 or B.shape[1]==0:
            raise ValueError("empty predictor block")
        inter=pd.DataFrame(index=tmp.index)
        for a in A.columns:
            for b in B.columns:
                inter[f"INT__{a}__X__{b}"]=A[a]*B[b]
        blocks=[A,B,inter]
        if adjustment_vars:
            for av in adjustment_vars:
                if av in (v1,v2): continue
                blocks.append(variable_block(tmp,av))
        X=pd.concat(blocks,axis=1)
        X=X.loc[:,~X.columns.duplicated()].astype(float)
        # Drop zero-variance columns before rank check.
        X=X.loc[:,X.nunique(dropna=False)>1]
        X=sm.add_constant(X,has_constant="add")
        rank=np.linalg.matrix_rank(X.to_numpy())
        full_rank=(rank==X.shape[1])
        if not full_rank:
            return {"var1":v1,"var2":v2,"n":len(tmp),"p_value":np.nan,"sparse_flag":sparse,"reliable":False,"reason":"rank deficient"}
        fit=sm.GLM(tmp[TARGET].astype(float),X,family=sm.families.Poisson()).fit(cov_type="HC0")
        int_cols=[c for c in X.columns if c.startswith("INT__")]
        if not int_cols:
            raise ValueError("no interaction columns")
        idx=[fit.params.index.get_loc(c) for c in int_cols]
        R=np.zeros((len(idx),len(fit.params)))
        for i,j in enumerate(idx): R[i,j]=1
        w=fit.wald_test(R,scalar=True)
        p=float(w.pvalue)
        reliable=(not sparse) and np.isfinite(p)
        return {"var1":v1,"var2":v2,"n":len(tmp),"p_value":p,"df_interaction":len(idx),
                "wald_chi2":float(w.statistic),"sparse_flag":sparse,"reliable":reliable,
                "reason":"" if reliable else "sparse categorical cells"}
    except Exception as e:
        return {"var1":v1,"var2":v2,"n":len(tmp),"p_value":np.nan,"sparse_flag":sparse,"reliable":False,"reason":str(e)[:180]}

In [18]:
# Pair-only screen.
pair_results = pd.DataFrame([fit_pair_screen(df,v1,v2,None) for v1,v2 in ALL_PAIRS])
# Full planned family is preserved: unreliable tests receive p=1 for BH-FDR.
p_for_fdr = pair_results["p_value"].where(pair_results["reliable"],1.0).fillna(1.0)
pair_results["q_value"] = multipletests(p_for_fdr, method="fdr_bh")[1]
pair_results["raw_p_lt_05"] = pair_results["reliable"] & pair_results["p_value"].lt(.05)
pair_results["fdr_q_lt_05"] = pair_results["reliable"] & pair_results["q_value"].lt(.05)

# Adjusted screen: background variables plus CRP/temperature unless they are the tested pair.
SCREEN_ADJUSTMENT_VARS = ADJUSTMENT_VARS + ["CRP","Body_Temperature"]
adjusted_results = pd.DataFrame([fit_pair_screen(df,v1,v2,SCREEN_ADJUSTMENT_VARS) for v1,v2 in ALL_PAIRS])
p_for_fdr_adj = adjusted_results["p_value"].where(adjusted_results["reliable"],1.0).fillna(1.0)
adjusted_results["q_value"] = multipletests(p_for_fdr_adj, method="fdr_bh")[1]
adjusted_results["raw_p_lt_05"] = adjusted_results["reliable"] & adjusted_results["p_value"].lt(.05)
adjusted_results["fdr_q_lt_05"] = adjusted_results["reliable"] & adjusted_results["q_value"].lt(.05)

screen_summary = pd.DataFrame([
    {"analysis":"Pair-only","planned_pairs":253,"reliable_tests":int(pair_results.reliable.sum()),"raw_p_lt_05":int(pair_results.raw_p_lt_05.sum()),"BH_FDR_q_lt_05":int(pair_results.fdr_q_lt_05.sum())},
    {"analysis":"Adjusted","planned_pairs":253,"reliable_tests":int(adjusted_results.reliable.sum()),"raw_p_lt_05":int(adjusted_results.raw_p_lt_05.sum()),"BH_FDR_q_lt_05":int(adjusted_results.fdr_q_lt_05.sum())},
])
display(screen_summary)

supported_both = pair_results[["var1","var2","q_value","fdr_q_lt_05","reliable"]].merge(
    adjusted_results[["var1","var2","q_value","fdr_q_lt_05","reliable"]], on=["var1","var2"], suffixes=("_pair","_adjusted")
)
supported_both = supported_both[
    supported_both.reliable_pair & supported_both.reliable_adjusted & supported_both.fdr_q_lt_05_pair & supported_both.fdr_q_lt_05_adjusted
].copy()

display(supported_both.sort_values("q_value_adjusted"))

pair_results.to_csv(TAB / "all_253_pairwise_interactions.csv", index=False)
adjusted_results.to_csv(TAB / "all_253_adjusted_interactions.csv", index=False)
supported_both.to_csv(TAB / "interactions_supported_in_both_253_screens.csv", index=False)
screen_summary.to_csv(TAB / "interaction_screen_summary_253.csv", index=False)

# Extract CRP x temperature within the full multiplicity-controlled family.
mask_pair = ((pair_results.var1.eq("CRP") & pair_results.var2.eq("Body_Temperature")) |
             (pair_results.var2.eq("CRP") & pair_results.var1.eq("Body_Temperature")))
mask_adj = ((adjusted_results.var1.eq("CRP") & adjusted_results.var2.eq("Body_Temperature")) |
            (adjusted_results.var2.eq("CRP") & adjusted_results.var1.eq("Body_Temperature")))
print("CRP x temperature in pair-only 253-family:")
display(pair_results.loc[mask_pair])
print("CRP x temperature in adjusted 253-family:")
display(adjusted_results.loc[mask_adj])

,analysis,planned_pairs,reliable_tests,raw_p_lt_05,BH_FDR_q_lt_05
0,Pair-only,253,183,27,4
1,Adjusted,253,167,18,0


,var1,var2,q_value_pair,fdr_q_lt_05_pair,reliable_pair,q_value_adjusted,fdr_q_lt_05_adjusted,reliable_adjusted


CRP x temperature in pair-only 253-family:


,var1,var2,n,p_value,df_interaction,wald_chi2,sparse_flag,reliable,reason,q_value,raw_p_lt_05,fdr_q_lt_05
47,Body_Temperature,CRP,766,0.037895,2.0,6.545865,False,True,,0.394761,True,False


CRP x temperature in adjusted 253-family:


,var1,var2,n,p_value,df_interaction,wald_chi2,sparse_flag,reliable,reason,q_value,raw_p_lt_05,fdr_q_lt_05
47,Body_Temperature,CRP,606,0.02831,2.0,7.129091,False,True,,0.651127,True,False


## 17. Out-of-sample prediction: logistic model without vs with CRP × temperature

Prediction is deliberately separated from inferential modeling. All preprocessing statistics are learned within each training fold to avoid leakage. The two models are evaluated on identical folds.

In [19]:
PRED_CONTINUOUS = ["Age","BMI","WBC_Count","Neutrophil_Percentage","log_CRP1","Body_Temperature"]
PRED_CATEGORICAL = ["Sex","Migratory_Pain","Lower_Right_Abd_Pain","Contralateral_Rebound_Tenderness",
                    "Coughing_Pain","Psoas_Sign","Nausea","Loss_of_Appetite","Dysuria","Peritonitis"]
PRED_REQUIRED = [TARGET] + PRED_CONTINUOUS + PRED_CATEGORICAL
pred_df = df[PRED_REQUIRED].dropna().copy()
print("Prediction sample n:",len(pred_df),"appendicitis:",int(pred_df[TARGET].sum()))

# sklearn compatibility across versions.
def make_encoder():
    try:
        return OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

def fold_design(train, valid, include_interaction):
    train=train.copy(); valid=valid.copy()
    means=train[PRED_CONTINUOUS].mean(); sds=train[PRED_CONTINUOUS].std(ddof=1).replace(0,1)
    Xtr_c=(train[PRED_CONTINUOUS]-means)/sds
    Xva_c=(valid[PRED_CONTINUOUS]-means)/sds
    # Explicit quadratic temperature based on fold-standardized T.
    Xtr_c["Body_Temperature_sq"] = Xtr_c["Body_Temperature"]**2
    Xva_c["Body_Temperature_sq"] = Xva_c["Body_Temperature"]**2
    if include_interaction:
        Xtr_c["CRP_x_T"] = Xtr_c["log_CRP1"]*Xtr_c["Body_Temperature"]
        Xva_c["CRP_x_T"] = Xva_c["log_CRP1"]*Xva_c["Body_Temperature"]
    enc=make_encoder(); Xtr_cat=enc.fit_transform(train[PRED_CATEGORICAL]); Xva_cat=enc.transform(valid[PRED_CATEGORICAL])
    names=list(enc.get_feature_names_out(PRED_CATEGORICAL))
    Xtr=pd.concat([Xtr_c.reset_index(drop=True),pd.DataFrame(Xtr_cat,columns=names)],axis=1)
    Xva=pd.concat([Xva_c.reset_index(drop=True),pd.DataFrame(Xva_cat,columns=names)],axis=1)
    return Xtr, Xva

def metric_bundle(y,p,threshold=.5):
    pred=(p>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        "AUROC":roc_auc_score(y,p),"AUPRC":average_precision_score(y,p),"Brier":brier_score_loss(y,p),
        "Accuracy":accuracy_score(y,pred),"Precision":precision_score(y,pred,zero_division=0),
        "Recall":recall_score(y,pred,zero_division=0),"Specificity":tn/(tn+fp) if tn+fp else np.nan,
        "F1":f1_score(y,pred,zero_division=0)
    }

def cv_oof_once(data, splitter):
    y=data[TARGET].to_numpy(); p0=np.full(len(data),np.nan); p1=np.full(len(data),np.nan)
    for tr,va in splitter.split(data,y):
        train=data.iloc[tr]; valid=data.iloc[va]
        Xtr0,Xva0=fold_design(train,valid,False); Xtr1,Xva1=fold_design(train,valid,True)
        m0=LogisticRegression(C=1e6,solver="lbfgs",max_iter=5000,random_state=RANDOM_STATE).fit(Xtr0,train[TARGET])
        m1=LogisticRegression(C=1e6,solver="lbfgs",max_iter=5000,random_state=RANDOM_STATE).fit(Xtr1,train[TARGET])
        p0[va]=m0.predict_proba(Xva0)[:,1]; p1[va]=m1.predict_proba(Xva1)[:,1]
    return y,p0,p1

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
y,p_base,p_int=cv_oof_once(pred_df,skf)
base_metrics=metric_bundle(y,p_base); int_metrics=metric_bundle(y,p_int)
cv_metrics=pd.DataFrame([{"model":"Without CRP x temperature",**base_metrics},{"model":"With CRP x temperature",**int_metrics}])
display(cv_metrics)
cv_metrics.to_csv(TAB / "fivefold_prediction_metrics.csv",index=False)

# Calibration intercept and slope from pooled out-of-fold predictions.
def calibration_intercept_slope(y, p):
    eps=1e-6
    p=np.clip(np.asarray(p),eps,1-eps)
    logit_p=np.log(p/(1-p))
    X=sm.add_constant(logit_p)
    res=sm.GLM(y,X,family=sm.families.Binomial()).fit()
    return {"calibration_intercept":float(res.params[0]),"calibration_slope":float(res.params[1])}

calibration_results=pd.DataFrame([
    {"model":"Without CRP x temperature", **calibration_intercept_slope(y,p_base)},
    {"model":"With CRP x temperature", **calibration_intercept_slope(y,p_int)},
])
display(calibration_results)
calibration_results.to_csv(TAB / "fivefold_oof_calibration.csv",index=False)

pd.DataFrame({"row_index":pred_df.index,"outcome":y,"p_without_interaction":p_base,"p_with_interaction":p_int}).to_csv(
    TAB / "fivefold_oof_predictions.csv", index=False
)


Prediction sample n: 606 appendicitis: 343


,model,AUROC,AUPRC,Brier,Accuracy,Precision,Recall,Specificity,F1
0,Without CRP x temperature,0.801317,0.844699,0.181164,0.724422,0.752874,0.763848,0.673004,0.758321
1,With CRP x temperature,0.802714,0.844652,0.180445,0.732673,0.760807,0.769679,0.684411,0.765217


,model,calibration_intercept,calibration_slope
0,Without CRP x temperature,0.020624,0.83683
1,With CRP x temperature,0.017209,0.83193


In [20]:
# Repeated 20 x 5-fold CV using RepeatedStratifiedKFold exactly once.
N_REPEATS=20
rskf=RepeatedStratifiedKFold(n_splits=5,n_repeats=N_REPEATS,random_state=RANDOM_STATE)
y=pred_df[TARGET].to_numpy()
repeat_predictions={r:{"base":np.full(len(pred_df),np.nan),"interaction":np.full(len(pred_df),np.nan)} for r in range(N_REPEATS)}

for split_i,(tr,va) in enumerate(rskf.split(pred_df,y)):
    rep=split_i//5
    train=pred_df.iloc[tr]; valid=pred_df.iloc[va]
    Xtr0,Xva0=fold_design(train,valid,False); Xtr1,Xva1=fold_design(train,valid,True)
    m0=LogisticRegression(C=1e6,solver="lbfgs",max_iter=5000,random_state=RANDOM_STATE).fit(Xtr0,train[TARGET])
    m1=LogisticRegression(C=1e6,solver="lbfgs",max_iter=5000,random_state=RANDOM_STATE).fit(Xtr1,train[TARGET])
    repeat_predictions[rep]["base"][va]=m0.predict_proba(Xva0)[:,1]
    repeat_predictions[rep]["interaction"][va]=m1.predict_proba(Xva1)[:,1]

repeat_rows=[]
for rep in range(N_REPEATS):
    p0=repeat_predictions[rep]["base"]; p1=repeat_predictions[rep]["interaction"]
    if np.isnan(p0).any() or np.isnan(p1).any():
        raise RuntimeError(f"Missing OOF predictions in repetition {rep+1}")
    m0=metric_bundle(y,p0); m1=metric_bundle(y,p1)
    row={"repetition":rep+1}
    for k in m0:
        row[f"base_{k}"]=m0[k]; row[f"interaction_{k}"]=m1[k]; row[f"delta_{k}"]=m1[k]-m0[k]
    repeat_rows.append(row)

repeated_cv=pd.DataFrame(repeat_rows)
repeated_cv.to_csv(TAB / "repeated_20x5fold_prediction_metrics.csv",index=False)

summary=[]
for metric in ["AUROC","AUPRC","Brier","Accuracy","Precision","Recall","Specificity","F1"]:
    d=repeated_cv[f"delta_{metric}"]
    better=(d<0).mean() if metric=="Brier" else (d>0).mean()
    summary.append({"metric":metric,"mean_delta":d.mean(),"sd_delta":d.std(ddof=1),"median_delta":d.median(),
                    "min_delta":d.min(),"max_delta":d.max(),"interaction_better_fraction":better})
repeated_summary=pd.DataFrame(summary)
display(repeated_summary)
repeated_summary.to_csv(TAB / "repeated_cv_delta_summary.csv",index=False)

,metric,mean_delta,sd_delta,median_delta,min_delta,max_delta,interaction_better_fraction
0,AUROC,0.000355,0.001208,0.000377,-0.001663,0.002117,0.55
1,AUPRC,-0.002086,0.001163,-0.002190,-0.003974,-0.000046,0.00
2,Brier,-0.000321,0.000650,-0.000498,-0.001134,0.000869,0.65
3,Accuracy,0.007591,0.004680,0.007426,0.000000,0.016502,0.95
4,Precision,0.005793,0.004708,0.006041,-0.001429,0.013251,0.85
5,Recall,0.008163,0.005483,0.005831,0.000000,0.017493,0.95
6,Specificity,0.006844,0.008146,0.005703,-0.007605,0.022814,0.70
7,F1,0.006947,0.004061,0.006745,0.000706,0.015080,1.00


## 18. Sensitivity analyses

The sections below probe assumptions behind the primary CRP–temperature estimate. They cover sample restriction, risk-scale interpretation, functional form, CRP transformation, missingness, ultrasound findings, and diagnostic prediction. These checks are exploratory and should be read alongside the primary model, rather than as independent confirmations.


### 18A. Same-sample comparison, robust uncertainty, and log-link risk bounds

This section separates the effect of covariate adjustment from the effect of changing the sample from 766 to 606 children. It also checks whether modified-Poisson fitted risks exceed 1 and adds a nonparametric bootstrap for the interaction coefficient.

In [21]:
# Same 606 children, but WITHOUT adjustment covariates in the formula.
poisson_primary_same606 = fit_modified_poisson(PRIMARY_FORMULA, adjusted_data)

# HC0 and HC3 are both requested here for transparency. In statsmodels GLM,
# HC0/HC3 may coincide numerically; the bootstrap below is therefore the key
# alternative uncertainty analysis.
def fit_poisson_cov(formula, data, cov_type):
    return smf.glm(formula=formula, data=data, family=sm.families.Poisson()).fit(cov_type=cov_type)

same_sample_rows = []
for label, formula, data in [
    ("Primary analysis sample", PRIMARY_FORMULA, primary_data),
    ("Same 606 children, unadjusted", PRIMARY_FORMULA, adjusted_data),
    ("Same 606 children, adjusted", ADJUSTED_FORMULA, adjusted_data),
]:
    for cov_type in ["HC0", "HC3"]:
        res = fit_poisson_cov(formula, data, cov_type)
        ci = res.conf_int().loc[term]
        pred = np.asarray(res.predict(data))
        same_sample_rows.append({
            "analysis": label,
            "covariance": cov_type,
            "n": int(res.nobs),
            "ratio_of_RRs": float(np.exp(res.params[term])),
            "CI_lower": float(np.exp(ci.iloc[0])),
            "CI_upper": float(np.exp(ci.iloc[1])),
            "p_value": float(res.pvalues[term]),
            "max_fitted_risk": float(np.nanmax(pred)),
            "n_fitted_risk_gt_1": int(np.sum(pred > 1)),
            "pct_fitted_risk_gt_1": float(100*np.mean(pred > 1)),
        })

same_sample_comparison = pd.DataFrame(same_sample_rows)
display(same_sample_comparison)
same_sample_comparison.to_csv(TAB / "analysis_same_sample_HC0_HC3.csv", index=False)

# Nonparametric bootstrap for the interaction ratio on the 606-child adjusted sample.
BOOT_REPS = 1000
rng = np.random.default_rng(RANDOM_STATE)
boot_beta = []
for b in range(BOOT_REPS):
    idx = rng.integers(0, len(adjusted_data), len(adjusted_data))
    boot = adjusted_data.iloc[idx].copy()
    try:
        r = smf.glm(ADJUSTED_FORMULA, data=boot, family=sm.families.Poisson()).fit()
        if term in r.params.index and np.isfinite(r.params[term]):
            boot_beta.append(float(r.params[term]))
    except Exception:
        pass

boot_beta = np.asarray(boot_beta)
bootstrap_summary = pd.DataFrame([{
    "requested_reps": BOOT_REPS,
    "successful_reps": len(boot_beta),
    "ratio_of_RRs_point_estimate": float(np.exp(poisson_crp_temperature_adjusted.params[term])),
    "bootstrap_CI_lower": float(np.exp(np.quantile(boot_beta, 0.025))),
    "bootstrap_CI_upper": float(np.exp(np.quantile(boot_beta, 0.975))),
}])
display(bootstrap_summary)
bootstrap_summary.to_csv(TAB / "analysis_bootstrap_interaction.csv", index=False)

,analysis,covariance,n,ratio_of_RRs,CI_lower,CI_upper,p_value,max_fitted_risk,n_fitted_risk_gt_1,pct_fitted_risk_gt_1
0,Primary analysis sample,HC0,766,0.922840,0.865814,0.983621,0.013609,1.299289,27,3.524804
1,Primary analysis sample,HC3,766,0.922840,0.865814,0.983621,0.013609,1.299289,27,3.524804
2,"Same 606 children, unadjusted",HC0,606,0.924694,0.852671,1.002801,0.058443,1.290838,17,2.805281
3,"Same 606 children, unadjusted",HC3,606,0.924694,0.852671,1.002801,0.058443,1.290838,17,2.805281
4,"Same 606 children, adjusted",HC0,606,0.919463,0.853952,0.989999,0.025981,1.560683,60,9.900990
5,"Same 606 children, adjusted",HC3,606,0.919463,0.853952,0.989999,0.025981,1.560683,60,9.900990


,requested_reps,successful_reps,ratio_of_RRs_point_estimate,bootstrap_CI_lower,bootstrap_CI_upper
0,1000,1000,0.919463,0.852279,0.99867


### 18B. Marginal-standardized risks and additive-scale interaction

To avoid relying only on the multiplicative risk-ratio scale, this section uses the adjusted logistic model to calculate bounded marginal risks under four standardized scenarios: observed CRP/temperature, doubled CRP, temperature increased by 1°C, and both changes together. The additive interaction contrast is the difference between the CRP-associated risk difference at the higher and original temperature. A marginal RERI-style contrast is also reported.

Because the original model uses `log(CRP+1)`, the scenario doubles CRP on its original mg/L scale and then re-applies `log(CRP+1)`.

In [22]:
def scenario_frame(base, crp_multiplier=1.0, temp_shift=0.0):
    x = base.copy()
    # Recover raw CRP and temperature from the original cleaned dataframe using row index.
    raw_crp = df.loc[x.index, "CRP"].astype(float)
    raw_temp = df.loc[x.index, "Body_Temperature"].astype(float)
    new_crp = raw_crp * crp_multiplier
    new_temp = raw_temp + temp_shift
    x["z_log_CRP1"] = (np.log1p(new_crp) - scaling["log_CRP1"]["mean"]) / scaling["log_CRP1"]["sd"]
    x["z_Body_Temperature"] = (new_temp - scaling["Body_Temperature"]["mean"]) / scaling["Body_Temperature"]["sd"]
    x["z_Body_Temperature_sq"] = x["z_Body_Temperature"]**2
    return x

scenarios = {
    "observed": scenario_frame(adjusted_data, 1.0, 0.0),
    "CRP doubled": scenario_frame(adjusted_data, 2.0, 0.0),
    "temperature +1C": scenario_frame(adjusted_data, 1.0, 1.0),
    "CRP doubled + temperature +1C": scenario_frame(adjusted_data, 2.0, 1.0),
}

# Use logistic regression for marginal risks so predictions remain in [0,1].
scenario_risks = {name: float(np.mean(logit_adjusted.predict(x))) for name, x in scenarios.items()}
p00 = scenario_risks["observed"]
p10 = scenario_risks["CRP doubled"]
p01 = scenario_risks["temperature +1C"]
p11 = scenario_risks["CRP doubled + temperature +1C"]

additive_interaction = p11 - p10 - p01 + p00
reri_marginal = (p11/p00) - (p10/p00) - (p01/p00) + 1

marginal_summary = pd.DataFrame([
    {"scenario": k, "marginal_risk": v} for k,v in scenario_risks.items()
])
marginal_contrasts = pd.DataFrame([{
    "CRP_RD_at_observed_temperature": p10-p00,
    "CRP_RD_after_plus1C": p11-p01,
    "additive_interaction_RD_difference": additive_interaction,
    "marginal_RERI_style_contrast": reri_marginal,
}])
display(marginal_summary)
display(marginal_contrasts)
marginal_summary.to_csv(TAB / "analysis_marginal_standardized_risks.csv", index=False)
marginal_contrasts.to_csv(TAB / "analysis_additive_interaction.csv", index=False)

# Bootstrap CIs for marginal contrasts.
MARGINAL_BOOT_REPS = 500
rng = np.random.default_rng(RANDOM_STATE + 1)
boot_rows = []
for b in range(MARGINAL_BOOT_REPS):
    idx = rng.integers(0, len(adjusted_data), len(adjusted_data))
    boot = adjusted_data.iloc[idx].copy()
    try:
        lr = smf.glm(ADJUSTED_FORMULA, data=boot, family=sm.families.Binomial()).fit()
        s00 = scenario_frame(boot,1,0); s10=scenario_frame(boot,2,0)
        s01 = scenario_frame(boot,1,1); s11=scenario_frame(boot,2,1)
        q00,q10,q01,q11 = [float(np.mean(lr.predict(z))) for z in [s00,s10,s01,s11]]
        boot_rows.append({
            "additive": q11-q10-q01+q00,
            "RERI": (q11/q00)-(q10/q00)-(q01/q00)+1,
        })
    except Exception:
        pass
boot_marginal = pd.DataFrame(boot_rows)
boot_marginal_ci = pd.DataFrame([
    {"measure":"Additive interaction (risk difference scale)",
     "estimate":additive_interaction,
     "CI_lower":boot_marginal["additive"].quantile(.025),
     "CI_upper":boot_marginal["additive"].quantile(.975)},
    {"measure":"Marginal RERI-style contrast",
     "estimate":reri_marginal,
     "CI_lower":boot_marginal["RERI"].quantile(.025),
     "CI_upper":boot_marginal["RERI"].quantile(.975)},
])
display(boot_marginal_ci)
boot_marginal_ci.to_csv(TAB / "analysis_additive_interaction_bootstrap.csv", index=False)

,scenario,marginal_risk
0,observed,0.566007
1,CRP doubled,0.593750
2,temperature +1C,0.503678
3,CRP doubled + temperature +1C,0.523327


,CRP_RD_at_observed_temperature,CRP_RD_after_plus1C,additive_interaction_RD_difference,marginal_RERI_style_contrast
0,0.027743,0.019648,-0.008095,-0.014301


,measure,estimate,CI_lower,CI_upper
0,Additive interaction (risk difference scale),-0.008095,-0.023715,0.007315
1,Marginal RERI-style contrast,-0.014301,-0.042153,0.012868


### 18C. Flexible functional-form analysis: additive smooths versus a joint CRP–temperature surface

The initial functional-form comparisons supported different specifications for CRP and temperature. This section applies one consistent comparison on the same 766 children. Logistic models with natural-cubic-spline main effects are compared with a model containing their full cross-product surface. The likelihood-ratio test asks whether the flexible joint surface improves fit beyond additive smooths of CRP and temperature.

In [23]:
flex_df = df[[TARGET, "log_CRP1", "Body_Temperature"]].dropna().copy()

flex_linear = smf.glm(
    f"{TARGET} ~ log_CRP1 + Body_Temperature",
    data=flex_df, family=sm.families.Binomial()
).fit()
flex_additive_smooth = smf.glm(
    f"{TARGET} ~ cr(log_CRP1, df=4) + cr(Body_Temperature, df=4)",
    data=flex_df, family=sm.families.Binomial()
).fit()
flex_joint_surface = smf.glm(
    f"{TARGET} ~ cr(log_CRP1, df=4) * cr(Body_Temperature, df=4)",
    data=flex_df, family=sm.families.Binomial()
).fit()

lr_add_vs_joint = 2*(flex_joint_surface.llf-flex_additive_smooth.llf)
df_add_vs_joint = int(round(flex_joint_surface.df_model-flex_additive_smooth.df_model))
p_add_vs_joint = chi2.sf(lr_add_vs_joint, df_add_vs_joint)

flex_compare = pd.DataFrame([
    {"model":"Linear additive", "n":len(flex_df), "df_model":flex_linear.df_model,
     "log_likelihood":flex_linear.llf, "AIC":flex_linear.aic, "BIC":flex_linear.bic_llf},
    {"model":"Additive natural cubic splines (4 df each)", "n":len(flex_df), "df_model":flex_additive_smooth.df_model,
     "log_likelihood":flex_additive_smooth.llf, "AIC":flex_additive_smooth.aic, "BIC":flex_additive_smooth.bic_llf},
    {"model":"Joint spline surface", "n":len(flex_df), "df_model":flex_joint_surface.df_model,
     "log_likelihood":flex_joint_surface.llf, "AIC":flex_joint_surface.aic, "BIC":flex_joint_surface.bic_llf},
])
display(flex_compare)
print(f"Additive smooths vs joint surface: LR chi2={lr_add_vs_joint:.3f}, df={df_add_vs_joint}, p={p_add_vs_joint:.6f}")
flex_compare.to_csv(TAB / "analysis_flexible_surface_models.csv", index=False)
pd.DataFrame([{"LR_chi2":lr_add_vs_joint,"df":df_add_vs_joint,"p_value":p_add_vs_joint}]).to_csv(
    TAB / "analysis_joint_surface_test.csv", index=False)

,model,n,df_model,log_likelihood,AIC,BIC
0,Linear additive,766,2,-453.227270,912.454541,926.378087
1,Additive natural cubic splines (4 df each),766,6,-445.440800,904.881601,937.369876
2,Joint spline surface,766,15,-439.149193,910.298386,984.557301


Additive smooths vs joint surface: LR chi2=12.583, df=9, p=0.182390


### 18D. Sensitivity to the constant in log(CRP + c)

The dataset contains exact CRP values of 0 mg/L. This section reports their frequency and repeats the same CRP×temperature model with several positive constants instead of fixing the transformation at `+1`. This does not identify the assay detection limit; that information must be verified from the source study or laboratory documentation.

In [24]:
crp_nonmissing = df["CRP"].dropna()
crp_zero_summary = pd.DataFrame([{
    "n_nonmissing_CRP": len(crp_nonmissing),
    "n_CRP_equal_0": int((crp_nonmissing==0).sum()),
    "pct_CRP_equal_0": float(100*(crp_nonmissing==0).mean()),
    "minimum_positive_CRP": float(crp_nonmissing[crp_nonmissing>0].min()) if (crp_nonmissing>0).any() else np.nan,
}])
display(crp_zero_summary)
crp_zero_summary.to_csv(TAB / "analysis_crp_zero_frequency.csv", index=False)

constant_rows=[]
for c in [0.1, 0.5, 1.0, 2.0, 5.0]:
    tmp=df.copy()
    tmp["crp_trans"] = np.log(tmp["CRP"] + c)
    tmp["z_crp_trans"] = (tmp["crp_trans"]-tmp["crp_trans"].mean())/tmp["crp_trans"].std(ddof=1)
    form = f"{TARGET} ~ z_crp_trans + z_Body_Temperature + z_Body_Temperature_sq + z_crp_trans:z_Body_Temperature"
    dat = tmp[[TARGET,"z_crp_trans","z_Body_Temperature","z_Body_Temperature_sq"]].dropna()
    r = smf.glm(form, data=dat, family=sm.families.Poisson()).fit(cov_type="HC0")
    t = "z_crp_trans:z_Body_Temperature"
    ci=r.conf_int().loc[t]
    constant_rows.append({"constant_c":c,"n":int(r.nobs),"ratio_of_RRs":np.exp(r.params[t]),
                          "CI_lower":np.exp(ci.iloc[0]),"CI_upper":np.exp(ci.iloc[1]),"p_value":r.pvalues[t]})
constant_sensitivity=pd.DataFrame(constant_rows)
display(constant_sensitivity)
constant_sensitivity.to_csv(TAB / "analysis_crp_constant_sensitivity.csv", index=False)

,n_nonmissing_CRP,n_CRP_equal_0,pct_CRP_equal_0,minimum_positive_CRP
0,771,167,21.660182,1.0


,constant_c,n,ratio_of_RRs,CI_lower,CI_upper,p_value
0,0.1,766,0.930657,0.856491,1.011246,0.089878
1,0.5,766,0.924187,0.862516,0.990268,0.025252
2,1.0,766,0.922840,0.865814,0.983621,0.013609
3,2.0,766,0.922341,0.869584,0.978298,0.007143
4,5.0,766,0.923066,0.875427,0.973298,0.003066


### 18E(i). Source data and measurement documentation

Source documentation is used to interpret recorded measurements and outcome definitions.

- Body temperature is documented as being measured with a thermometer placed **in the rectum or in the auditory canal**.
- The public documentation identifies CRP in **mg/L**, but an assay platform and analytical detection limit were not identified in the available dataset metadata or associated papers reviewed .
- Therefore, exact CRP values of 0 mg/L are described as **recorded zeros** and are not assumed to represent values below a known detection limit.
- The diagnosis variable is not a purely histologic outcome: operated patients were histologically confirmed, whereas conservatively managed patients could be labeled as appendicitis using clinical-score and ultrasound criteria. This is important when interpreting ultrasound-adjusted sensitivity analyses.

These documentation details provide context for interpreting the observed measurements and diagnosis labels.


### 18E(ii). Multiple-imputation sensitivity analysis

To address missing-data sensitivity without treating categorical variables as continuous measurements, this analysis uses a simple fully conditional stochastic imputation procedure.

- Continuous variables are imputed by predictive mean matching from observed donor values.
- Binary categorical variables are imputed from logistic-regression probabilities.
- Peritonitis is imputed with multinomial logistic regression.
- Diagnosis is fully observed and is included as a predictor but is never imputed.
- CRP is imputed on the `log(CRP + 1)` scale and transformed back only when needed.
- Twenty completed datasets are generated after iterative cycling through the incomplete variables.
- The adjusted modified-Poisson and logistic CRP×temperature coefficients are pooled using Rubin's rules.

This is a sensitivity analysis rather than a replacement for the complete-case primary analyses.


In [25]:
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MI_M = 20
MI_BURN = 10
PMM_K = 5

# Work on the variables required by the adjusted model.
# CRP is represented on the log(CRP+1) scale during imputation.
mi_cont = ["Age", "BMI", "WBC_Count", "Neutrophil_Percentage",
           "log_CRP1", "Body_Temperature"]
mi_binary = [
    "Sex", "Migratory_Pain", "Lower_Right_Abd_Pain",
    "Contralateral_Rebound_Tenderness", "Coughing_Pain",
    "Psoas_Sign", "Nausea", "Loss_of_Appetite", "Dysuria"
]
mi_multiclass = ["Peritonitis"]
mi_vars = mi_cont + mi_binary + mi_multiclass

mi_base = df[[TARGET] + mi_vars].copy()

# Stable observed category levels.
binary_levels = {
    v: list(pd.Series(mi_base[v].dropna().unique()).astype(str).sort_values())
    for v in mi_binary
}
multi_levels = {
    v: list(pd.Series(mi_base[v].dropna().unique()).astype(str).sort_values())
    for v in mi_multiclass
}

def _initial_fill(frame):
    out = frame.copy()
    for v in mi_cont:
        out[v] = pd.to_numeric(out[v], errors="coerce")
        out[v] = out[v].fillna(out[v].median())
    for v in mi_binary + mi_multiclass:
        mode = out[v].mode(dropna=True)
        fill = mode.iloc[0] if len(mode) else ""
        out[v] = out[v].fillna(fill).astype(str)
    return out

def _design_matrix(frame, exclude_var):
    """Numerical design matrix from the current completed data."""
    pred = frame[[c for c in [TARGET] + mi_vars if c != exclude_var]].copy()
    # Make categorical predictors explicit rather than integer-coding them.
    pred = pd.get_dummies(
        pred,
        columns=[c for c in mi_binary + mi_multiclass if c != exclude_var],
        drop_first=False,
        dtype=float
    )
    return pred.astype(float)

def _pmm_impute(current, original, var, rng, k=5):
    miss = original[var].isna()
    if not miss.any():
        return
    obs = ~miss
    X = _design_matrix(current, var)
    y = pd.to_numeric(original.loc[obs, var], errors="coerce").astype(float)

    # Bootstrap the observed rows so different imputations include model uncertainty.
    obs_idx = np.flatnonzero(obs.to_numpy())
    boot_idx = rng.choice(obs_idx, size=len(obs_idx), replace=True)

    model = BayesianRidge()
    model.fit(X.iloc[boot_idx], pd.to_numeric(current.iloc[boot_idx][var], errors="coerce"))

    pred_obs = model.predict(X.loc[obs])
    pred_miss, pred_sd = model.predict(X.loc[miss], return_std=True)
    pred_miss = rng.normal(pred_miss, np.maximum(pred_sd, 1e-8))

    donor_vals = y.to_numpy()
    for row_index, pm in zip(current.index[miss], pred_miss):
        d = np.abs(pred_obs - pm)
        kk = min(k, len(d))
        donor_positions = np.argpartition(d, kk - 1)[:kk]
        chosen = rng.choice(donor_positions)
        current.at[row_index, var] = donor_vals[chosen]

def _binary_impute(current, original, var, rng):
    miss = original[var].isna()
    if not miss.any():
        return
    obs = ~miss
    X = _design_matrix(current, var)
    levels = binary_levels[var]
    if len(levels) != 2:
        raise ValueError(f"{var} expected 2 observed levels, found {levels}")
    y = original.loc[obs, var].astype(str).map({levels[0]: 0, levels[1]: 1}).astype(int)

    obs_idx = np.flatnonzero(obs.to_numpy())
    boot_idx = rng.choice(obs_idx, size=len(obs_idx), replace=True)
    y_boot = original.iloc[boot_idx][var].astype(str).map({levels[0]: 0, levels[1]: 1}).astype(int)

    # If a rare bootstrap sample loses a class, use all observed rows.
    fit_idx = boot_idx if y_boot.nunique() == 2 else obs_idx
    y_fit = original.iloc[fit_idx][var].astype(str).map({levels[0]: 0, levels[1]: 1}).astype(int)

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, C=1e6, solver="lbfgs")
    )
    model.fit(X.iloc[fit_idx], y_fit)
    p = model.predict_proba(X.loc[miss])[:, 1]
    draw = rng.binomial(1, p)
    current.loc[miss, var] = np.where(draw == 1, levels[1], levels[0])

def _multiclass_impute(current, original, var, rng):
    miss = original[var].isna()
    if not miss.any():
        return
    obs = ~miss
    X = _design_matrix(current, var)
    levels = multi_levels[var]
    level_to_int = {lev: i for i, lev in enumerate(levels)}

    obs_idx = np.flatnonzero(obs.to_numpy())
    boot_idx = rng.choice(obs_idx, size=len(obs_idx), replace=True)
    y_boot = original.iloc[boot_idx][var].astype(str).map(level_to_int).astype(int)

    fit_idx = boot_idx if y_boot.nunique() == len(levels) else obs_idx
    y_fit = original.iloc[fit_idx][var].astype(str).map(level_to_int).astype(int)

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, C=1e6, solver="lbfgs")
    )
    model.fit(X.iloc[fit_idx], y_fit)
    probs = model.predict_proba(X.loc[miss])
    classes = model.named_steps["logisticregression"].classes_

    sampled = []
    for pr in probs:
        sampled.append(rng.choice(classes, p=pr / pr.sum()))
    inv = {i: lev for lev, i in level_to_int.items()}
    current.loc[miss, var] = [inv[int(x)] for x in sampled]

def make_imputed_dataset(seed):
    rng = np.random.default_rng(seed)
    current = _initial_fill(mi_base)

    for cycle in range(MI_BURN):
        # Random scan order avoids always favoring the same variable ordering.
        order = list(mi_cont + mi_binary + mi_multiclass)
        rng.shuffle(order)
        for v in order:
            if v in mi_cont:
                _pmm_impute(current, mi_base, v, rng, k=PMM_K)
            elif v in mi_binary:
                _binary_impute(current, mi_base, v, rng)
            else:
                _multiclass_impute(current, mi_base, v, rng)

    # Recreate the terms used by the analysis models.
    current["CRP"] = np.expm1(pd.to_numeric(current["log_CRP1"], errors="coerce")).clip(lower=0)

    for v in ["log_CRP1", "Body_Temperature"]:
        current[f"z_{v}"] = (
            current[v].astype(float) - current[v].astype(float).mean()
        ) / current[v].astype(float).std(ddof=1)
    current["z_Body_Temperature_sq"] = current["z_Body_Temperature"] ** 2
    current[TARGET] = mi_base[TARGET].astype(int).values
    return current

mi_poisson = []
mi_logistic = []

for m in range(MI_M):
    imp_df = make_imputed_dataset(RANDOM_STATE + 1000 + m)

    rp = smf.glm(
        ADJUSTED_FORMULA,
        data=imp_df,
        family=sm.families.Poisson()
    ).fit(cov_type="HC3")

    rl = smf.glm(
        ADJUSTED_FORMULA,
        data=imp_df,
        family=sm.families.Binomial()
    ).fit(cov_type="HC3")

    mi_poisson.append({
        "imputation": m + 1,
        "beta": float(rp.params[term]),
        "variance": float(rp.cov_params().loc[term, term])
    })
    mi_logistic.append({
        "imputation": m + 1,
        "beta": float(rl.params[term]),
        "variance": float(rl.cov_params().loc[term, term])
    })

def rubin_pool(component_df, label):
    """Pool one scalar coefficient across imputations using Rubin's rules.

    Confidence intervals and p-values use Rubin's finite-m t degrees of freedom
    rather than a normal approximation. This is used as a sensitivity analysis.
    """
    d = pd.DataFrame(component_df).copy()
    m = len(d)
    if m < 2:
        raise ValueError("Rubin pooling requires at least two imputations.")

    qbar = d["beta"].mean()
    ubar = d["variance"].mean()
    b = d["beta"].var(ddof=1)
    tvar = ubar + (1 + 1 / m) * b
    se = np.sqrt(tvar)

    # Rubin's large-sample degrees of freedom for scalar pooling.
    # If between-imputation variance is effectively zero, df -> infinity.
    if b <= 0 or not np.isfinite(b):
        r = 0.0
        df_rubin = np.inf
    elif ubar <= 0 or not np.isfinite(ubar):
        r = np.inf
        df_rubin = m - 1
    else:
        r = ((1 + 1 / m) * b) / ubar
        df_rubin = (m - 1) * (1 + 1 / r) ** 2

    crit = 1.96 if not np.isfinite(df_rubin) else scipy.stats.t.ppf(0.975, df=df_rubin)
    stat = qbar / se
    if np.isfinite(df_rubin):
        p_value = 2 * scipy.stats.t.sf(abs(stat), df=df_rubin)
    else:
        p_value = 2 * scipy.stats.norm.sf(abs(stat))

    pooled = pd.DataFrame([{
        "analysis": label,
        "m": m,
        "pooled_exp_beta": np.exp(qbar),
        "CI_lower": np.exp(qbar - crit * se),
        "CI_upper": np.exp(qbar + crit * se),
        "test_statistic": stat,
        "df_rubin": df_rubin,
        "p_value": p_value,
        "within_imputation_variance": ubar,
        "between_imputation_variance": b,
        "relative_increase_variance": r,
        "total_variance": tvar,
    }])
    return pooled, d

mi_poisson_pooled, mi_poisson_components = rubin_pool(
    mi_poisson, "Modified Poisson (HC3)"
)
mi_logistic_pooled, mi_logistic_components = rubin_pool(
    mi_logistic, "Logistic regression (HC3)"
)

mi_pooled = pd.concat(
    [mi_poisson_pooled, mi_logistic_pooled],
    ignore_index=True
)

display(mi_pooled)

mi_poisson_components.to_csv(
    TAB / "analysis_multiple_imputation_poisson_components.csv", index=False
)
mi_logistic_components.to_csv(
    TAB / "analysis_multiple_imputation_logistic_components.csv", index=False
)
mi_pooled.to_csv(
    TAB / "analysis_multiple_imputation_pooled.csv", index=False
)


,analysis,m,pooled_exp_beta,CI_lower,CI_upper,test_statistic,df_rubin,p_value,within_imputation_variance,between_imputation_variance,relative_increase_variance,total_variance
0,Modified Poisson (HC3),20,0.909995,0.857179,0.966065,-3.091829,19469.614547,0.001992,0.000901,0.000028,0.032246,0.000931
1,Logistic regression (HC3),20,0.750550,0.596121,0.944985,-2.441493,33987.337763,0.014632,0.013487,0.000311,0.024216,0.013813


### 18F. Ultrasound availability and imaging sensitivity analyses

Ultrasound was performed in most children, so the analysis distinguishes **ultrasound availability** from **appendix visualization**.

Three sensitivity analyses are used:

1. the CRP×temperature model is repeated in the subgroup with ultrasound performed;
2. appendix visualization and free intraperitoneal fluid are added on the same ultrasound-complete sample;
3. appendix visualization and diameter are combined into an explicit ultrasound appendix-status variable so that a non-visualized appendix is treated as an informative category rather than as a missing diameter. A separate visualized-appendix analysis evaluates continuous diameter only where it is actually observed.

These analyses are sensitivity checks because ultrasound findings are closely related to the diagnostic process and, for some conservatively managed patients, appendix diameter contributed to the dataset's diagnosis definition.


In [26]:
# ------------------------------------------------------------
# Availability
# ------------------------------------------------------------
us_summary = []
for v in ["US_Performed", "Appendix_on_US", "Appendix_Diameter", "Free_Fluids"]:
    us_summary.append({
        "variable": v,
        "nonmissing_n": int(df[v].notna().sum()),
        "missing_n": int(df[v].isna().sum()),
        "missing_pct": float(100 * df[v].isna().mean())
    })

us_availability = pd.DataFrame(us_summary)
display(us_availability)
us_availability.to_csv(
    TAB / "analysis_ultrasound_availability.csv", index=False
)

print("US performed:")
display(df["US_Performed"].value_counts(dropna=False).rename("n").to_frame())

print("Appendix visualized on US:")
display(df["Appendix_on_US"].value_counts(dropna=False).rename("n").to_frame())

# Cross-tab explicitly showing why diameter is structurally unavailable.
diameter_availability_by_visibility = pd.crosstab(
    df["Appendix_on_US"].fillna("missing visualization"),
    df["Appendix_Diameter"].notna().map({True: "diameter recorded", False: "diameter missing"}),
    dropna=False
)
display(diameter_availability_by_visibility)
diameter_availability_by_visibility.to_csv(
    TAB / "analysis_diameter_availability_by_visualization.csv"
)

# ------------------------------------------------------------
# Define imaged subgroup
# ------------------------------------------------------------
imaged = df[
    df["US_Performed"].astype("string").str.strip().str.lower().eq("yes")
].copy()

rows = []

# Main model in imaged subgroup.
imaged_primary = imaged[
    [TARGET, "z_log_CRP1", "z_Body_Temperature", "z_Body_Temperature_sq"]
].dropna()
imaged_adj = imaged[adjusted_required].dropna()

ri0 = fit_modified_poisson(PRIMARY_FORMULA, imaged_primary)
ri1 = fit_modified_poisson(ADJUSTED_FORMULA, imaged_adj)

for label, r in [
    ("Imaged subgroup, CRP and temperature", ri0),
    ("Imaged subgroup, adjusted", ri1)
]:
    ci = r.conf_int().loc[term]
    rows.append({
        "analysis": label,
        "n": int(r.nobs),
        "ratio_of_RRs": np.exp(r.params[term]),
        "CI_lower": np.exp(ci.iloc[0]),
        "CI_upper": np.exp(ci.iloc[1]),
        "p_value": r.pvalues[term]
    })

# ------------------------------------------------------------
# Same-sample addition of visualization and free fluid
# ------------------------------------------------------------
us_extra_required = adjusted_required + ["Appendix_on_US", "Free_Fluids"]
us_extra = imaged[us_extra_required].dropna().copy()

us_base = fit_modified_poisson(ADJUSTED_FORMULA, us_extra)
us_formula = ADJUSTED_FORMULA + " + C(Appendix_on_US) + C(Free_Fluids)"
us_model = fit_modified_poisson(us_formula, us_extra)

for label, r in [
    ("Same US-complete sample, no US findings", us_base),
    ("Same US-complete sample, + visualization/free fluid", us_model)
]:
    ci = r.conf_int().loc[term]
    rows.append({
        "analysis": label,
        "n": int(r.nobs),
        "ratio_of_RRs": np.exp(r.params[term]),
        "CI_lower": np.exp(ci.iloc[0]),
        "CI_upper": np.exp(ci.iloc[1]),
        "p_value": r.pvalues[term]
    })

# ------------------------------------------------------------
# Explicit appendix-status variable:
# non-visualized / visualized <6 mm / visualized >=6 mm
# ------------------------------------------------------------
imaged["US_appendix_status"] = np.nan
vis = imaged["Appendix_on_US"].astype("string").str.strip().str.lower()

imaged.loc[vis.eq("no"), "US_appendix_status"] = "not visualized"

vis_yes = vis.eq("yes")
diameter = pd.to_numeric(imaged["Appendix_Diameter"], errors="coerce")
imaged.loc[vis_yes & diameter.notna() & diameter.lt(6), "US_appendix_status"] = "visualized <6 mm"
imaged.loc[vis_yes & diameter.notna() & diameter.ge(6), "US_appendix_status"] = "visualized >=6 mm"
imaged.loc[vis_yes & diameter.isna(), "US_appendix_status"] = "visualized, diameter missing"

print("Composite ultrasound appendix status:")
display(imaged["US_appendix_status"].value_counts(dropna=False).rename("n").to_frame())

status_required = adjusted_required + ["US_appendix_status", "Free_Fluids"]
status_df = imaged[status_required].dropna().copy()

status_base = fit_modified_poisson(ADJUSTED_FORMULA, status_df)
status_formula = (
    ADJUSTED_FORMULA
    + " + C(US_appendix_status, Treatment(reference='not visualized'))"
    + " + C(Free_Fluids)"
)
status_model = fit_modified_poisson(status_formula, status_df)

for label, r in [
    ("Same appendix-status sample, no US findings", status_base),
    ("Same appendix-status sample, + appendix status/free fluid", status_model)
]:
    ci = r.conf_int().loc[term]
    rows.append({
        "analysis": label,
        "n": int(r.nobs),
        "ratio_of_RRs": np.exp(r.params[term]),
        "CI_lower": np.exp(ci.iloc[0]),
        "CI_upper": np.exp(ci.iloc[1]),
        "p_value": r.pvalues[term]
    })

# ------------------------------------------------------------
# Continuous diameter only where appendix was visualized
# ------------------------------------------------------------
visualized = imaged.loc[
    vis.eq("yes") & pd.to_numeric(imaged["Appendix_Diameter"], errors="coerce").notna()
].copy()
visualized["Appendix_Diameter"] = pd.to_numeric(
    visualized["Appendix_Diameter"], errors="coerce"
)

diam_required = adjusted_required + ["Appendix_Diameter"]
diam = visualized[diam_required].dropna().copy()

diam_base = fit_modified_poisson(ADJUSTED_FORMULA, diam)
diam_model = fit_modified_poisson(
    ADJUSTED_FORMULA + " + Appendix_Diameter",
    diam
)

for label, r in [
    ("Visualized-appendix sample, no diameter", diam_base),
    ("Visualized-appendix sample, + continuous diameter", diam_model)
]:
    ci = r.conf_int().loc[term]
    rows.append({
        "analysis": label,
        "n": int(r.nobs),
        "ratio_of_RRs": np.exp(r.params[term]),
        "CI_lower": np.exp(ci.iloc[0]),
        "CI_upper": np.exp(ci.iloc[1]),
        "p_value": r.pvalues[term]
    })

ultrasound_sensitivity = pd.DataFrame(rows)
display(ultrasound_sensitivity)

ultrasound_sensitivity.to_csv(
    TAB / "analysis_ultrasound_sensitivity.csv", index=False
)

imaged["US_appendix_status"].value_counts(dropna=False).rename("n").to_csv(
    TAB / "analysis_ultrasound_appendix_status_counts.csv"
)


,variable,nonmissing_n,missing_n,missing_pct
0,US_Performed,777,3,0.384615
1,Appendix_on_US,777,3,0.384615
2,Appendix_Diameter,498,282,36.153846
3,Free_Fluids,719,61,7.820513


US performed:


,n
US_Performed,
yes,763
no,14
NaN,3


Appendix visualized on US:


,n
Appendix_on_US,
yes,504
no,273
NaN,3


Appendix_Diameter,diameter missing,diameter recorded
Appendix_on_US,,
missing visualization,3,0
no,269,4
yes,10,494


Composite ultrasound appendix status:


C:\Users\Asus\AppData\Local\Temp\ipykernel_9800\3305572882.py:99: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not visualized' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  imaged.loc[vis.eq("no"), "US_appendix_status"] = "not visualized"


,n
US_appendix_status,
visualized >=6 mm,380
not visualized,262
visualized <6 mm,112
"visualized, diameter missing",8
NaN,1


,analysis,n,ratio_of_RRs,CI_lower,CI_upper,p_value
0,"Imaged subgroup, CRP and temperature",752,0.924414,0.866342,0.986379,0.017584
1,"Imaged subgroup, adjusted",599,0.927764,0.858955,1.002084,0.056520
2,"Same US-complete sample, no US findings",561,0.929061,0.857492,1.006603,0.072009
3,"Same US-complete sample, + visualization/free ...",561,0.957081,0.878503,1.042687,0.315562
4,"Same appendix-status sample, no US findings",561,0.929061,0.857492,1.006603,0.072009
5,"Same appendix-status sample, + appendix status...",561,1.026843,0.955047,1.104037,0.473828
6,"Visualized-appendix sample, no diameter",393,0.871026,0.805336,0.942075,0.000558
7,"Visualized-appendix sample, + continuous diameter",393,0.919445,0.849473,0.995181,0.037564


### 18G. Diagnostic framing: CRP discrimination across temperature strata

Temperature is grouped using clinically interpretable categories: <37.0°C, 37.0–37.9°C, and ≥38.0°C. Within each stratum, CRP AUROC is reported with bootstrap confidence intervals. Sensitivity, specificity, positive likelihood ratio, and negative likelihood ratio are also reported for CRP thresholds of 10 and 50 mg/L. A logistic interaction test between transformed CRP and temperature stratum provides a formal global test of whether the CRP–diagnosis relationship differs across these strata.

In [27]:
dx = df[[TARGET,"CRP","log_CRP1","Body_Temperature"]].dropna().copy()
dx["temp_stratum"] = pd.cut(dx["Body_Temperature"], bins=[-np.inf,37.0,38.0,np.inf], right=False,
                            labels=["<37.0°C","37.0–37.9°C","≥38.0°C"])

AUC_BOOT=1000
rng=np.random.default_rng(RANDOM_STATE+2)
auc_rows=[]
for level,g in dx.groupby("temp_stratum", observed=True):
    auc=roc_auc_score(g[TARGET],g["CRP"])
    boots=[]
    for b in range(AUC_BOOT):
        samp=g.iloc[rng.integers(0,len(g),len(g))]
        if samp[TARGET].nunique()==2:
            boots.append(roc_auc_score(samp[TARGET],samp["CRP"]))
    auc_rows.append({"temperature_stratum":str(level),"n":len(g),"appendicitis_n":int(g[TARGET].sum()),
                     "CRP_AUROC":auc,"CI_lower":np.quantile(boots,.025),"CI_upper":np.quantile(boots,.975)})
auc_by_temp=pd.DataFrame(auc_rows)
display(auc_by_temp)
auc_by_temp.to_csv(TAB / "analysis_crp_auc_by_temperature.csv",index=False)

# Diagnostic operating characteristics at literature-relevant CRP thresholds.
def diagnostic_at_threshold(g, thr):
    y=g[TARGET].to_numpy(); pred=(g["CRP"].to_numpy()>=thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    sens=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    plr=sens/(1-spec) if np.isfinite(spec) and spec<1 else np.inf
    nlr=(1-sens)/spec if spec>0 else np.inf
    return sens,spec,plr,nlr

dx_rows=[]
for level,g in dx.groupby("temp_stratum",observed=True):
    for thr in [10,50]:
        sens,spec,plr,nlr=diagnostic_at_threshold(g,thr)
        dx_rows.append({"temperature_stratum":str(level),"CRP_threshold_mg_L":thr,"n":len(g),
                        "sensitivity":sens,"specificity":spec,"LR_positive":plr,"LR_negative":nlr})
diagnostic_by_temp=pd.DataFrame(dx_rows)
display(diagnostic_by_temp)
diagnostic_by_temp.to_csv(TAB / "analysis_crp_diagnostic_by_temperature.csv",index=False)

# Formal global interaction test on the logit scale.
base_strata = smf.glm(f"{TARGET} ~ log_CRP1 + C(temp_stratum)",data=dx,family=sm.families.Binomial()).fit()
int_strata = smf.glm(f"{TARGET} ~ log_CRP1 * C(temp_stratum)",data=dx,family=sm.families.Binomial()).fit()
lr=2*(int_strata.llf-base_strata.llf)
df_diff=int(round(int_strata.df_model-base_strata.df_model))
p_global=chi2.sf(lr,df_diff)
strata_test=pd.DataFrame([{"LR_chi2":lr,"df":df_diff,"p_value":p_global}])
display(strata_test)
strata_test.to_csv(TAB / "analysis_crp_temperature_strata_interaction_test.csv",index=False)

,temperature_stratum,n,appendicitis_n,CRP_AUROC,CI_lower,CI_upper
0,<37.0°C,235,116,0.717763,0.647865,0.784251
1,37.0–37.9°C,343,207,0.747442,0.698238,0.798246
2,≥38.0°C,188,130,0.670889,0.589076,0.756269


,temperature_stratum,CRP_threshold_mg_L,n,sensitivity,specificity,LR_positive,LR_negative
0,<37.0°C,10,235,0.362069,0.873950,2.872414,0.729940
1,<37.0°C,50,235,0.120690,0.983193,7.181034,0.894341
2,37.0–37.9°C,10,343,0.570048,0.757353,2.349290,0.567703
3,37.0–37.9°C,50,343,0.217391,0.955882,4.927536,0.818729
4,≥38.0°C,10,188,0.823077,0.379310,1.326068,0.466434
5,≥38.0°C,50,188,0.446154,0.706897,1.522172,0.783490


,LR_chi2,df,p_value
0,1.470225,2,0.479452


### 18H. Saturation check using other inflammatory-marker interactions

To assess whether the CRP–temperature pattern is specific to temperature or reflects a more general ceiling/saturation phenomenon among correlated inflammatory markers, the same 606-child sample is used to compare CRP×temperature, CRP×WBC, and CRP×neutrophil interactions.

In [28]:
sat = adjusted_data.copy()
# Add standardized WBC and neutrophils using the same 606-child sample.
for v in ["WBC_Count","Neutrophil_Percentage"]:
    sat[f"z_{v}"]=(sat[v]-sat[v].mean())/sat[v].std(ddof=1)

base_other=[t for t in ADJUSTMENT_TERMS if t not in ["WBC_Count","Neutrophil_Percentage"]]
marker_models={
    "CRP x temperature": ADJUSTED_FORMULA,
    "CRP x WBC": f"{TARGET} ~ z_log_CRP1 + z_WBC_Count + z_log_CRP1:z_WBC_Count + " + " + ".join(base_other + ["Neutrophil_Percentage","z_Body_Temperature","z_Body_Temperature_sq"]),
    "CRP x neutrophils": f"{TARGET} ~ z_log_CRP1 + z_Neutrophil_Percentage + z_log_CRP1:z_Neutrophil_Percentage + " + " + ".join(base_other + ["WBC_Count","z_Body_Temperature","z_Body_Temperature_sq"]),
}
marker_terms={
    "CRP x temperature":"z_log_CRP1:z_Body_Temperature",
    "CRP x WBC":"z_log_CRP1:z_WBC_Count",
    "CRP x neutrophils":"z_log_CRP1:z_Neutrophil_Percentage",
}
marker_rows=[]
for name,form in marker_models.items():
    r=smf.glm(form,data=sat,family=sm.families.Poisson()).fit(cov_type="HC0")
    t=marker_terms[name]; ci=r.conf_int().loc[t]; pred=np.asarray(r.predict(sat))
    marker_rows.append({"interaction":name,"n":int(r.nobs),"ratio_of_RRs":np.exp(r.params[t]),
                        "CI_lower":np.exp(ci.iloc[0]),"CI_upper":np.exp(ci.iloc[1]),"p_value":r.pvalues[t],
                        "max_fitted_risk":pred.max(),"n_fitted_risk_gt_1":int((pred>1).sum())})
marker_comparison=pd.DataFrame(marker_rows)
display(marker_comparison)
marker_comparison.to_csv(TAB / "analysis_inflammatory_interaction_comparison.csv",index=False)

,interaction,n,ratio_of_RRs,CI_lower,CI_upper,p_value,max_fitted_risk,n_fitted_risk_gt_1
0,CRP x temperature,606,0.919463,0.853952,0.989999,0.025981,1.560683,60
1,CRP x WBC,606,0.913665,0.860440,0.970183,0.003194,1.520995,60
2,CRP x neutrophils,606,0.869415,0.797459,0.947865,0.001500,1.522074,53


### 18I. Support at higher temperatures

This section quantifies how many children support inference in the upper temperature range and avoids treating visually extrapolated portions of the interaction curve as equally precise.

In [29]:
temp_support=[]
for cutoff in [38.0,38.5,39.0,39.5,40.0]:
    x=adjusted_data["z_Body_Temperature"]*scaling["Body_Temperature"]["sd"]+scaling["Body_Temperature"]["mean"]
    mask=x>=cutoff
    temp_support.append({"cutoff_C":cutoff,"n_at_or_above":int(mask.sum()),
                         "appendicitis_n":int(adjusted_data.loc[mask,TARGET].sum()),
                         "no_appendicitis_n":int(mask.sum()-adjusted_data.loc[mask,TARGET].sum())})
temp_support=pd.DataFrame(temp_support)
display(temp_support)
temp_support.to_csv(TAB / "analysis_high_temperature_support.csv",index=False)

,cutoff_C,n_at_or_above,appendicitis_n,no_appendicitis_n
0,38.0,139,93,46
1,38.5,65,38,27
2,39.0,30,17,13
3,39.5,11,6,5
4,40.0,1,1,0


### 18J. Prediction calibration and repeated-CV nuance

The original notebook already produced calibration intercept and slope for the pooled 5-fold out-of-fold predictions. Here, calibration is also summarized across the 20 repeated 5-fold runs so the prediction section can report discrimination, overall accuracy, and calibration together.

In [30]:
cal_rows=[]
for rep in range(N_REPEATS):
    for label,key in [("Without interaction","base"),("With interaction","interaction")]:
        p=repeat_predictions[rep][key]
        c=calibration_intercept_slope(y,p)
        cal_rows.append({"repetition":rep+1,"model":label,**c})
repeated_calibration=pd.DataFrame(cal_rows)
calibration_summary=(repeated_calibration.groupby("model")[["calibration_intercept","calibration_slope"]]
                     .agg(["mean","std","min","max"]))
display(calibration_summary)
repeated_calibration.to_csv(TAB / "analysis_repeated_cv_calibration.csv",index=False)

calibration_intercept                                \
                                     mean       std       min       max   
model                                                                     
With interaction                 0.019055  0.011565 -0.000647  0.052618   
Without interaction              0.018511  0.011247 -0.003098  0.050831   

                    calibration_slope                                
                                 mean       std       min       max  
model                                                                
With interaction             0.828537  0.025044  0.782084  0.863454  
Without interaction          0.834290  0.023865  0.776961  0.868031

## 19. Analysis output checks

The following checks confirm that the main analyses and sensitivity analyses were produced.

In [31]:
# Dataset and analysis structure checks.
assert len(df) == 780
assert int(df[TARGET].sum()) == 463
assert len(CANDIDATE_PREDICTORS) == 23
assert len(ALL_PAIRS) == 253
assert len(pair_results) == 253 and len(adjusted_results) == 253

# Main model estimates.
primary_row=primary_table.loc[primary_table.term.eq(term)].iloc[0]
adj_row=adjusted_table.loc[adjusted_table.term.eq(term)].iloc[0]
assert int(primary_row["n"]) == 766
assert int(adj_row["n"]) == 606
assert np.isclose(primary_row["estimate_exp"], 0.922840, atol=5e-4)
assert np.isclose(adj_row["estimate_exp"], 0.919463, atol=5e-4)

analysis_files = [
    TAB/"analysis_same_sample_HC0_HC3.csv",
    TAB/"analysis_bootstrap_interaction.csv",
    TAB/"analysis_marginal_standardized_risks.csv",
    TAB/"analysis_additive_interaction.csv",
    TAB/"analysis_flexible_surface_models.csv",
    TAB/"analysis_joint_surface_test.csv",
    TAB/"analysis_crp_zero_frequency.csv",
    TAB/"analysis_crp_constant_sensitivity.csv",
    TAB/"analysis_multiple_imputation_pooled.csv",
    TAB/"analysis_multiple_imputation_poisson_components.csv",
    TAB/"analysis_multiple_imputation_logistic_components.csv",
    TAB/"analysis_ultrasound_availability.csv",
    TAB/"analysis_diameter_availability_by_visualization.csv",
    TAB/"analysis_ultrasound_appendix_status_counts.csv",
    TAB/"analysis_ultrasound_sensitivity.csv",
    TAB/"analysis_crp_auc_by_temperature.csv",
    TAB/"analysis_crp_diagnostic_by_temperature.csv",
    TAB/"analysis_crp_temperature_strata_interaction_test.csv",
    TAB/"analysis_inflammatory_interaction_comparison.csv",
    TAB/"analysis_high_temperature_support.csv",
    TAB/"analysis_repeated_cv_calibration.csv",
]
missing=[str(p) for p in analysis_files if not p.exists()]
if missing:
    raise AssertionError(f"Missing analysis outputs: {missing}")

print("OUTPUT CONSISTENCY CHECKS PASSED")
print("Original 766-child ratio of RRs:", primary_row["estimate_exp"], "p=", primary_row["p_value"])
print("Adjusted 606-child ratio of RRs:", adj_row["estimate_exp"], "p=", adj_row["p_value"])

OUTPUT CONSISTENCY CHECKS PASSED
Original 766-child ratio of RRs: 0.9228397867097494 p= 0.013609005437977717
Adjusted 606-child ratio of RRs: 0.9194627974057689 p= 0.02598066135303384


## 20. Interpretation of statistical findings

- Treat the study as exploratory and diagnostic rather than causal; prefer language such as **"does the diagnostic association of CRP differ across temperature?"** over "effect modification" when possible.
- Report the modified-Poisson quantity as a **ratio of risk ratios**, not as a generic "interaction factor".
- Give logistic and modified-Poisson analyses comparable prominence; do not treat the modified-Poisson p-value as uniquely confirmatory.
- Explicitly distinguish the 1-df linear CRP×temperature test from the 2-df temperature interaction used in the 253-pair screen.
- Interpret the CRP×temperature result in light of any fitted risks above 1, the additive/marginal analyses, and the comparison with CRP×WBC and CRP×neutrophils.
- Do not infer a biological mechanism from the interaction because symptom duration, antipyretic use, and temperature measurement route are not available in the current analysis.
- Do not interpret the high-temperature tail of the curve as equally supported when few children contribute data there.
- The CRP assay detection limit cannot be inferred from exact zeros in the public dataset; verify assay details from source documentation before making claims about censoring.
- Ultrasound remains excluded from the main predictor family, but its availability and imaged-subgroup sensitivity analyses must now be reported.
- Prediction should be described as showing very small and mixed metric changes, with calibration reported alongside discrimination.

## 21. Additional interaction and nonlinear-model checks

The following analyses provide additional checks:
1. identify the four pair-only interactions that survive BH-FDR;
2. add a penalized tensor-product spline sensitivity analysis;
3. assemble a consolidated robustness summary table.


### 21A. Pair-only interactions after FDR correction


In [32]:
pair_only_fdr_hits = (
    pair_results.loc[
        pair_results["reliable"] & pair_results["fdr_q_lt_05"],
        ["var1","var2","n","df_interaction","wald_chi2","p_value","q_value"]
    ]
    .sort_values(["q_value","p_value"])
    .reset_index(drop=True)
)

display(pair_only_fdr_hits)

assert len(pair_only_fdr_hits) == 4, (
    f"Expected 4 pair-only BH-FDR-significant interactions, found {len(pair_only_fdr_hits)}"
)

pair_only_fdr_hits.to_csv(
    TAB / "analysis_pair_only_BH_FDR_hits.csv",
    index=False
)


,var1,var2,n,df_interaction,wald_chi2,p_value,q_value
0,WBC_Count,CRP,771,1.0,25.127809,5.365352e-07,0.000136
1,CRP,Peritonitis,766,2.0,22.640178,1.212685e-05,0.001534
2,WBC_Count,Peritonitis,769,2.0,17.887895,1.305248e-04,0.011008
3,Coughing_Pain,Psoas_Sign,738,1.0,12.012587,5.284246e-04,0.033423


### 21B. Penalized tensor-product spline sensitivity analysis

This analysis complements the unpenalized spline-surface likelihood-ratio test in Section 18C.
Natural cubic-style spline bases are estimated within each outer training fold. A penalized
logistic model containing separate CRP and temperature spline bases is compared with a model
that additionally contains all tensor-product basis terms. The ridge penalty strength is selected
inside each training fold by 3-fold cross-validation. Performance is then evaluated in the held-out
outer fold. This avoids using the test fold to choose either the spline transformation or penalty.


In [33]:
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score

pen_df = df[[TARGET, "log_CRP1", "Body_Temperature"]].dropna().copy()
X_pen = pen_df[["log_CRP1", "Body_Temperature"]].to_numpy(float)
y_pen = pen_df[TARGET].to_numpy(int)

def _tensor_basis(B1, B2):
    return np.einsum("ij,ik->ijk", B1, B2).reshape(len(B1), -1)

def _fit_penalized_logistic_inner_cv(X_train, y_train, X_test, y_test, seed):
    candidate_C = [0.01, 0.1, 1.0, 10.0]
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)

    best_C = None
    best_loss = np.inf

    for C in candidate_C:
        fold_losses = []
        for inner_train, inner_valid in inner.split(X_train, y_train):
            scaler = StandardScaler()
            Xi_train = scaler.fit_transform(X_train[inner_train])
            Xi_valid = scaler.transform(X_train[inner_valid])

            model = LogisticRegression(
                C=C,
                solver="lbfgs",
                max_iter=3000
            )
            model.fit(Xi_train, y_train[inner_train])
            p = model.predict_proba(Xi_valid)[:, 1]
            fold_losses.append(log_loss(y_train[inner_valid], p))

        mean_loss = float(np.mean(fold_losses))
        if mean_loss < best_loss:
            best_loss = mean_loss
            best_C = C

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    final_model = LogisticRegression(
        C=best_C,
        solver="lbfgs",
        max_iter=3000
    )
    final_model.fit(Xtr, y_train)
    p_test = final_model.predict_proba(Xte)[:, 1]

    return {
        "C": best_C,
        "log_loss": log_loss(y_test, p_test),
        "AUROC": roc_auc_score(y_test, p_test),
    }

outer = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=RANDOM_STATE + 77
)

tensor_rows = []

for split_id, (train_idx, test_idx) in enumerate(outer.split(X_pen, y_pen), start=1):

    crp_spline = SplineTransformer(
        n_knots=5,
        degree=3,
        include_bias=False
    )
    temp_spline = SplineTransformer(
        n_knots=5,
        degree=3,
        include_bias=False
    )

    Bc_train = crp_spline.fit_transform(X_pen[train_idx][:, [0]])
    Bt_train = temp_spline.fit_transform(X_pen[train_idx][:, [1]])

    Bc_test = crp_spline.transform(X_pen[test_idx][:, [0]])
    Bt_test = temp_spline.transform(X_pen[test_idx][:, [1]])

    X_add_train = np.c_[Bc_train, Bt_train]
    X_add_test = np.c_[Bc_test, Bt_test]

    X_joint_train = np.c_[
        Bc_train,
        Bt_train,
        _tensor_basis(Bc_train, Bt_train)
    ]
    X_joint_test = np.c_[
        Bc_test,
        Bt_test,
        _tensor_basis(Bc_test, Bt_test)
    ]

    for label, X_train, X_test in [
        ("Additive penalized splines", X_add_train, X_add_test),
        ("Penalized tensor-product surface", X_joint_train, X_joint_test),
    ]:
        metrics = _fit_penalized_logistic_inner_cv(
            X_train,
            y_pen[train_idx],
            X_test,
            y_pen[test_idx],
            seed=1000 + split_id
        )

        tensor_rows.append({
            "split": split_id,
            "model": label,
            **metrics
        })

penalized_tensor_cv = pd.DataFrame(tensor_rows)

tensor_model_summary = (
    penalized_tensor_cv
    .groupby("model")[["log_loss","AUROC"]]
    .agg(["mean","std"])
)

wide = penalized_tensor_cv.pivot(
    index="split",
    columns="model",
    values=["log_loss","AUROC"]
)

delta_log_loss = (
    wide["log_loss"]["Penalized tensor-product surface"]
    - wide["log_loss"]["Additive penalized splines"]
)

delta_auc = (
    wide["AUROC"]["Penalized tensor-product surface"]
    - wide["AUROC"]["Additive penalized splines"]
)

penalized_tensor_summary = pd.DataFrame([{
    "n": len(pen_df),
    "outer_splits": len(delta_log_loss),
    "mean_delta_log_loss_joint_minus_additive": delta_log_loss.mean(),
    "sd_delta_log_loss": delta_log_loss.std(ddof=1),
    "joint_lower_log_loss_pct": 100 * (delta_log_loss < 0).mean(),
    "mean_delta_AUROC_joint_minus_additive": delta_auc.mean(),
    "sd_delta_AUROC": delta_auc.std(ddof=1),
    "joint_higher_AUROC_pct": 100 * (delta_auc > 0).mean(),
}])

display(tensor_model_summary)
display(penalized_tensor_summary)

penalized_tensor_cv.to_csv(
    TAB / "analysis_penalized_tensor_nestedcv_results.csv",
    index=False
)

penalized_tensor_summary.to_csv(
    TAB / "analysis_penalized_tensor_nestedcv_summary.csv",
    index=False
)


log_loss               AUROC          
                                      mean       std      mean       std
model                                                                   
Additive penalized splines        0.589197  0.022728  0.733337  0.030334
Penalized tensor-product surface  0.595012  0.021154  0.731508  0.029323

,n,outer_splits,mean_delta_log_loss_joint_minus_additive,sd_delta_log_loss,joint_lower_log_loss_pct,mean_delta_AUROC_joint_minus_additive,sd_delta_AUROC,joint_higher_AUROC_pct
0,766,25,0.005814,0.005734,16.0,-0.001829,0.007593,36.0


### 21C. Summary of robustness estimates

This table is intended as the single summary of how the CRP--temperature finding behaves under
the main model and the additional sensitivity analyses. It does not force all analyses onto
one effect scale; instead, the `Result` column reports the scale appropriate to each analysis.


In [34]:
# Helper accessors
primary_hc0 = same_sample_comparison.query(
    "analysis == 'Primary analysis sample' and covariance == 'HC0'"
).iloc[0]

same606_hc0 = same_sample_comparison.query(
    "analysis == 'Same 606 children, unadjusted' and covariance == 'HC0'"
).iloc[0]

adjusted_hc0 = same_sample_comparison.query(
    "analysis == 'Same 606 children, adjusted' and covariance == 'HC0'"
).iloc[0]

logit_adj = logistic_sensitivity.query(
    "analysis == 'Multivariable adjusted'"
).iloc[0]

pair_crptemp_adj = adjusted_results.loc[
    ((adjusted_results["var1"] == "CRP") & (adjusted_results["var2"] == "Body_Temperature"))
    | ((adjusted_results["var2"] == "CRP") & (adjusted_results["var1"] == "Body_Temperature"))
].iloc[0]

additive_row = boot_marginal_ci.loc[
    boot_marginal_ci["measure"] == "Additive interaction (risk difference scale)"
].iloc[0]

mi_pois = mi_pooled.loc[
    mi_pooled["analysis"].str.startswith("Modified Poisson")
].iloc[0]

mi_logit = mi_pooled.loc[
    mi_pooled["analysis"].str.startswith("Logistic regression")
].iloc[0]

us_status = ultrasound_sensitivity.loc[
    ultrasound_sensitivity["analysis"].str.contains(
        r"\+ appendix status/free fluid",
        regex=True
    )
].iloc[0]

strata_row = strata_test.iloc[0]

tensor_row = penalized_tensor_summary.iloc[0]

robustness_rows = [
    {
        "Analysis": "Primary modified Poisson",
        "n": int(primary_hc0["n"]),
        "Result": (
            f"Ratio of RRs {primary_hc0['ratio_of_RRs']:.3f} "
            f"(95% CI {primary_hc0['CI_lower']:.3f}-{primary_hc0['CI_upper']:.3f}); "
            f"p={primary_hc0['p_value']:.3f}"
        ),
        "Interpretation": "Multiplicative interaction detected"
    },
    {
        "Analysis": "Same 606 children, unadjusted",
        "n": int(same606_hc0["n"]),
        "Result": (
            f"Ratio of RRs {same606_hc0['ratio_of_RRs']:.3f} "
            f"(95% CI {same606_hc0['CI_lower']:.3f}-{same606_hc0['CI_upper']:.3f}); "
            f"p={same606_hc0['p_value']:.3f}"
        ),
        "Interpretation": "Point estimate similar; conventional significance lost"
    },
    {
        "Analysis": "Adjusted modified Poisson",
        "n": int(adjusted_hc0["n"]),
        "Result": (
            f"Ratio of RRs {adjusted_hc0['ratio_of_RRs']:.3f} "
            f"(95% CI {adjusted_hc0['CI_lower']:.3f}-{adjusted_hc0['CI_upper']:.3f}); "
            f"p={adjusted_hc0['p_value']:.3f}"
        ),
        "Interpretation": "Multiplicative interaction detected"
    },
    {
        "Analysis": "Adjusted modified Poisson, bootstrap",
        "n": 606,
        "Result": (
            f"Ratio of RRs {bootstrap_summary.iloc[0]['ratio_of_RRs_point_estimate']:.3f}; "
            f"bootstrap 95% CI "
            f"{bootstrap_summary.iloc[0]['bootstrap_CI_lower']:.3f}-"
            f"{bootstrap_summary.iloc[0]['bootstrap_CI_upper']:.3f}"
        ),
        "Interpretation": "Borderline bootstrap interval"
    },
    {
        "Analysis": "Adjusted logistic regression",
        "n": int(logit_adj["n"]),
        "Result": (
            f"Interaction OR {logit_adj['interaction_OR']:.3f} "
            f"(95% CI {logit_adj['CI_lower']:.3f}-{logit_adj['CI_upper']:.3f}); "
            f"p={logit_adj['p_value']:.3f}"
        ),
        "Interpretation": "No conventional statistical significance"
    },
    {
        "Analysis": "253-pair adjusted 2-df temperature test",
        "n": int(pair_crptemp_adj["n"]),
        "Result": (
            f"p={pair_crptemp_adj['p_value']:.3f}; "
            f"BH-FDR q={pair_crptemp_adj['q_value']:.3f}"
        ),
        "Interpretation": "Does not survive multiple-testing correction"
    },
    {
        "Analysis": "Additive risk-scale interaction",
        "n": 606,
        "Result": (
            f"Risk-difference interaction {additive_row['estimate']:.4f} "
            f"(bootstrap 95% CI {additive_row['CI_lower']:.4f}-"
            f"{additive_row['CI_upper']:.4f})"
        ),
        "Interpretation": "No clear additive-scale interaction"
    },
    {
        "Analysis": "Flexible joint spline surface",
        "n": int(len(flex_df)),
        "Result": (
            f"Additive smooths vs joint surface: "
            f"LR chi2={lr_add_vs_joint:.2f}, df={df_add_vs_joint}, "
            f"p={p_add_vs_joint:.3f}"
        ),
        "Interpretation": "Joint surface not required beyond separate smooth effects"
    },
    {
        "Analysis": "Penalized tensor-product spline surface",
        "n": int(tensor_row["n"]),
        "Result": (
            f"Mean delta log loss={tensor_row['mean_delta_log_loss_joint_minus_additive']:.4f}; "
            f"mean delta AUROC={tensor_row['mean_delta_AUROC_joint_minus_additive']:.4f}"
        ),
        "Interpretation": "Joint surface did not improve held-out performance"
    },
    {
        "Analysis": "Multiple imputation: modified Poisson",
        "n": 780,
        "Result": (
            f"Ratio of RRs {mi_pois['pooled_exp_beta']:.3f} "
            f"(95% CI {mi_pois['CI_lower']:.3f}-{mi_pois['CI_upper']:.3f}); "
            f"p={mi_pois['p_value']:.3f}"
        ),
        "Interpretation": "Multiplicative interaction detected after imputation"
    },
    {
        "Analysis": "Multiple imputation: logistic",
        "n": 780,
        "Result": (
            f"Interaction OR {mi_logit['pooled_exp_beta']:.3f} "
            f"(95% CI {mi_logit['CI_lower']:.3f}-{mi_logit['CI_upper']:.3f}); "
            f"p={mi_logit['p_value']:.3f}"
        ),
        "Interpretation": "Logistic interaction detected after imputation"
    },
    {
        "Analysis": "Ultrasound-informed sensitivity",
        "n": int(us_status["n"]),
        "Result": (
            f"Ratio of RRs {us_status['ratio_of_RRs']:.3f} "
            f"(95% CI {us_status['CI_lower']:.3f}-{us_status['CI_upper']:.3f}); "
            f"p={us_status['p_value']:.3f}"
        ),
        "Interpretation": "Interaction attenuated after adding ultrasound information"
    },
    {
        "Analysis": "CRP discrimination across temperature strata",
        "n": int(len(dx)),
        "Result": (
            f"Global interaction LR chi2={strata_row['LR_chi2']:.2f}, "
            f"df={int(strata_row['df'])}, p={strata_row['p_value']:.3f}"
        ),
        "Interpretation": "No evidence that CRP discrimination differs by temperature stratum"
    },
    {
        "Analysis": "Prediction: repeated cross-validation",
        "n": 606,
        "Result": (
            f"Mean delta AUROC="
            f"{repeated_summary.loc[repeated_summary['metric']=='AUROC','mean_delta'].iloc[0]:+.5f}; "
            f"mean delta AUPRC="
            f"{repeated_summary.loc[repeated_summary['metric']=='AUPRC','mean_delta'].iloc[0]:+.5f}"
        ),
        "Interpretation": "No meaningful overall predictive improvement"
    },
]

master_robustness_table = pd.DataFrame(robustness_rows)

display(master_robustness_table)

master_robustness_table.to_csv(
    TAB / "analysis_master_robustness_table.csv",
    index=False
)


,Analysis,n,Result,Interpretation
0,Primary modified Poisson,766,Ratio of RRs 0.923 (95% CI 0.866-0.984); p=0.014,Multiplicative interaction detected
1,"Same 606 children, unadjusted",606,Ratio of RRs 0.925 (95% CI 0.853-1.003); p=0.058,Point estimate similar; conventional significa...
2,Adjusted modified Poisson,606,Ratio of RRs 0.919 (95% CI 0.854-0.990); p=0.026,Multiplicative interaction detected
3,"Adjusted modified Poisson, bootstrap",606,Ratio of RRs 0.919; bootstrap 95% CI 0.852-0.999,Borderline bootstrap interval
4,Adjusted logistic regression,606,Interaction OR 0.822 (95% CI 0.635-1.064); p=0...,No conventional statistical significance
5,253-pair adjusted 2-df temperature test,606,p=0.028; BH-FDR q=0.651,Does not survive multiple-testing correction
6,Additive risk-scale interaction,606,Risk-difference interaction -0.0081 (bootstrap...,No clear additive-scale interaction
7,Flexible joint spline surface,766,Additive smooths vs joint surface: LR chi2=12....,Joint surface not required beyond separate smo...
8,Penalized tensor-product spline surface,766,Mean delta log loss=0.0058; mean delta AUROC=-...,Joint surface did not improve held-out perform...
9,Multiple imputation: modified Poisson,780,Ratio of RRs 0.910 (95% CI 0.857-0.966); p=0.002,Multiplicative interaction detected after impu...


## 22. Further diagnostic and sensitivity analyses

These analyses examine precision, bounded-risk estimation,
outcome verification, missing data, diagnostic discrimination, flexible main effects,
the broader 253-pair screen, and the large number of recorded CRP values at 0 mg/L.

The goal is to judge consistency from **effect estimates and confidence intervals**, not from
whether an individual $p$-value falls above or below 0.05.

### 22A. Management type and feasibility of a histology-only sensitivity analysis

The public documentation states that operated patients were histologically confirmed, whereas
conservatively managed patients could receive the appendicitis label using clinical scores and ultrasound.
Because fever contributes to some clinical scores, management type is cross-tabulated against diagnosis
and the surgically managed subset is inspected as the potential histology-only sample.

A diagnostic interaction model is fitted only if this subset contains adequate numbers in both outcome
groups. If the surgically managed sample contains too few children without appendicitis, the notebook
reports the analysis as non-estimable rather than producing an unstable separation-driven estimate.

In [35]:
# Management by diagnosis
management_tab = pd.crosstab(
    df["Management"].fillna("missing"),
    df["Diagnosis"],
    margins=True
)
display(management_tab)
management_tab.to_csv(TAB / "sensitivity_management_by_diagnosis.csv")

# Operated patients are identified from the documented management labels.
management_norm = df["Management"].astype("string").str.strip().str.lower()
surgical_mask = (
    management_norm.str.contains("surgical", na=False)
    | management_norm.str.contains("appendectomy", na=False)
)
surgical_df = df.loc[surgical_mask].copy()

surg_outcome_counts = (
    surgical_df[TARGET]
    .value_counts()
    .reindex([0, 1], fill_value=0)
)
print("Surgically managed / histologically verified subset:", len(surgical_df))
print(
    "No appendicitis:", int(surg_outcome_counts.loc[0]),
    "| Appendicitis:", int(surg_outcome_counts.loc[1])
)

histology_feasibility = pd.DataFrame([{
    "n_surgical": len(surgical_df),
    "appendicitis_n": int(surg_outcome_counts.loc[1]),
    "no_appendicitis_n": int(surg_outcome_counts.loc[0]),
    "interaction_model_estimable": bool(surg_outcome_counts.min() >= 5),
}])

display(histology_feasibility)
histology_feasibility.to_csv(
    TAB / "sensitivity_histology_subset_feasibility.csv",
    index=False
)

# A diagnostic interaction requires adequate variation in the outcome.
# If either diagnosis group contains fewer than 5 children, the model is not
# estimated because the resulting interaction would be dominated by sparse
# outcome support / separation.
if surg_outcome_counts.min() < 5:
    histology_sensitivity = pd.DataFrame([{
        "analysis": "Surgically managed / histologically verified subset",
        "n": len(surgical_df),
        "appendicitis_n": int(surg_outcome_counts.loc[1]),
        "no_appendicitis_n": int(surg_outcome_counts.loc[0]),
        "estimable": False,
        "reason": "Too few children without appendicitis for a stable diagnostic interaction model",
    }])
    print(
        "Histology-only CRP x temperature model not estimated:",
        "the surgically managed subset has insufficient outcome variation."
    )
else:
    surg_required = [TARGET, "log_CRP1", "Body_Temperature"] + ADJUSTMENT_VARS
    surg_cc = surgical_df[surg_required].dropna().copy()

    surg_cc["z_log_CRP1"] = (
        surg_cc["log_CRP1"] - scaling["log_CRP1"]["mean"]
    ) / scaling["log_CRP1"]["sd"]
    surg_cc["z_Body_Temperature"] = (
        surg_cc["Body_Temperature"] - scaling["Body_Temperature"]["mean"]
    ) / scaling["Body_Temperature"]["sd"]
    surg_cc["z_Body_Temperature_sq"] = surg_cc["z_Body_Temperature"] ** 2

    surg_primary = smf.glm(
        PRIMARY_FORMULA,
        data=surg_cc,
        family=sm.families.Poisson()
    ).fit(cov_type="HC0")

    surg_adjusted = smf.glm(
        ADJUSTED_FORMULA,
        data=surg_cc,
        family=sm.families.Poisson()
    ).fit(cov_type="HC0")

    surg_logit = smf.glm(
        ADJUSTED_FORMULA,
        data=surg_cc,
        family=sm.families.Binomial()
    ).fit(cov_type="HC0")

    rows = []
    for label, model, metric in [
        ("Histology subset: CRP and temperature", surg_primary, "Ratio of risk ratios"),
        ("Histology subset: adjusted modified Poisson", surg_adjusted, "Ratio of risk ratios"),
        ("Histology subset: adjusted logistic", surg_logit, "Interaction OR"),
    ]:
        ci = model.conf_int().loc[term]
        rows.append({
            "analysis": label,
            "n": int(model.nobs),
            "measure": metric,
            "estimate": float(np.exp(model.params[term])),
            "CI_lower": float(np.exp(ci.iloc[0])),
            "CI_upper": float(np.exp(ci.iloc[1])),
            "p_value": float(model.pvalues[term]),
            "estimable": True,
        })

    histology_sensitivity = pd.DataFrame(rows)

display(histology_sensitivity)
histology_sensitivity.to_csv(
    TAB / "sensitivity_histology_only_sensitivity.csv",
    index=False
)

Diagnosis,appendicitis,no appendicitis,All
Management,,,
conservative,170,313,483
primary surgical,268,2,270
secondary surgical,25,1,26
simultaneous appendectomy,0,1,1
All,463,317,780


Surgically managed / histologically verified subset: 297
No appendicitis: 4 | Appendicitis: 293


,n_surgical,appendicitis_n,no_appendicitis_n,interaction_model_estimable
0,297,293,4,False


Histology-only CRP x temperature model not estimated: the surgically managed subset has insufficient outcome variation.


,analysis,n,appendicitis_n,no_appendicitis_n,estimable,reason
0,Surgically managed / histologically verified s...,297,293,4,False,Too few children without appendicitis for a st...


### 22B. Bounded-risk marginal standardization

Modified Poisson regression estimates relative risks directly, but some fitted values exceeded 1.
This analysis therefore uses the adjusted logistic model to obtain bounded predicted risks and
marginally standardized risk ratios.

For each child, CRP is doubled on its original mg/L scale and temperature is either left unchanged
or increased by 1$^\circ$C. The CRP-associated marginal risk ratio is calculated in both scenarios.
Their ratio answers the same clinical question as the multiplicative interaction while keeping all
predicted risks between 0 and 1.

The corresponding risk differences are also reported so the additive-scale contrast can be judged
against the size of the baseline CRP-associated risk difference.

In [36]:
def bounded_standardized_contrasts(base, fitted_logit):
    s00 = scenario_frame(base, 1.0, 0.0)
    s10 = scenario_frame(base, 2.0, 0.0)
    s01 = scenario_frame(base, 1.0, 1.0)
    s11 = scenario_frame(base, 2.0, 1.0)

    p00, p10, p01, p11 = [
        float(np.mean(fitted_logit.predict(z)))
        for z in [s00, s10, s01, s11]
    ]

    rr_crp_T = p10 / p00
    rr_crp_Tplus1 = p11 / p01
    ratio_rr = rr_crp_Tplus1 / rr_crp_T

    rd_crp_T = p10 - p00
    rd_crp_Tplus1 = p11 - p01
    rd_interaction = rd_crp_Tplus1 - rd_crp_T

    return {
        "risk_observed": p00,
        "risk_CRP_doubled": p10,
        "risk_temp_plus1C": p01,
        "risk_CRP_doubled_temp_plus1C": p11,
        "RR_for_CRP_doubling_at_observed_T": rr_crp_T,
        "RR_for_CRP_doubling_after_plus1C": rr_crp_Tplus1,
        "ratio_of_standardized_RRs": ratio_rr,
        "RD_for_CRP_doubling_at_observed_T": rd_crp_T,
        "RD_for_CRP_doubling_after_plus1C": rd_crp_Tplus1,
        "additive_interaction_RD_difference": rd_interaction,
    }

bounded_point = bounded_standardized_contrasts(adjusted_data, logit_adjusted)

BOUNDED_BOOT_REPS = 1000
rng = np.random.default_rng(RANDOM_STATE + 2201)
bounded_boot = []
bounded_boot_failures = 0

for b in range(BOUNDED_BOOT_REPS):
    sample_idx = rng.integers(0, len(adjusted_data), len(adjusted_data))
    boot = adjusted_data.iloc[sample_idx].copy()

    try:
        fit_b = smf.glm(
            ADJUSTED_FORMULA,
            data=boot,
            family=sm.families.Binomial()
        ).fit()
        bounded_boot.append(bounded_standardized_contrasts(boot, fit_b))
    except (ValueError, np.linalg.LinAlgError, RuntimeError) as exc:
        bounded_boot_failures += 1
        continue

print(f"Bounded bootstrap fits: {len(bounded_boot)}/{BOUNDED_BOOT_REPS} successful; "
      f"{bounded_boot_failures} failed")
if len(bounded_boot) < 0.95 * BOUNDED_BOOT_REPS:
    raise RuntimeError("Too many bootstrap fits failed; inspect separation and model convergence.")

bounded_boot = pd.DataFrame(bounded_boot)

bounded_rows = []
for measure in [
    "RR_for_CRP_doubling_at_observed_T",
    "RR_for_CRP_doubling_after_plus1C",
    "ratio_of_standardized_RRs",
    "RD_for_CRP_doubling_at_observed_T",
    "RD_for_CRP_doubling_after_plus1C",
    "additive_interaction_RD_difference",
]:
    bounded_rows.append({
        "measure": measure,
        "estimate": bounded_point[measure],
        "CI_lower": bounded_boot[measure].quantile(0.025),
        "CI_upper": bounded_boot[measure].quantile(0.975),
        "bootstrap_reps_successful": len(bounded_boot),
    })

bounded_standardized_results = pd.DataFrame(bounded_rows)
display(bounded_standardized_results)
bounded_standardized_results.to_csv(
    TAB / "sensitivity_bounded_standardized_interaction.csv",
    index=False
)

Bounded bootstrap fits: 1000/1000 successful; 0 failed


,measure,estimate,CI_lower,CI_upper,bootstrap_reps_successful
0,RR_for_CRP_doubling_at_observed_T,1.049015,1.029609,1.070480,1000
1,RR_for_CRP_doubling_after_plus1C,1.039010,0.997715,1.093085,1000
2,ratio_of_standardized_RRs,0.990462,0.961629,1.025745,1000
3,RD_for_CRP_doubling_at_observed_T,0.027743,0.017063,0.038944,1000
4,RD_for_CRP_doubling_after_plus1C,0.019648,-0.001240,0.041844,1000
5,additive_interaction_RD_difference,-0.008095,-0.022531,0.007228,1000


### 22C. Clinically interpretable contrasts: 1$^\circ$C and CRP doubling

Clinically interpretable contrasts summarize changes rather than relying only on 1-SD changes.
These estimates use the adjusted logistic model and marginal standardization, so they remain bounded
probabilities and do not require interpreting an impossible fitted risk above 1.

In [37]:
clinical_scale_effects = pd.DataFrame([
    {
        "contrast": "CRP doubling at observed temperature",
        "risk_ratio": bounded_point["RR_for_CRP_doubling_at_observed_T"],
        "risk_difference": bounded_point["RD_for_CRP_doubling_at_observed_T"],
    },
    {
        "contrast": "CRP doubling after temperature +1°C",
        "risk_ratio": bounded_point["RR_for_CRP_doubling_after_plus1C"],
        "risk_difference": bounded_point["RD_for_CRP_doubling_after_plus1C"],
    },
    {
        "contrast": "Ratio of CRP-doubling risk ratios after +1°C",
        "risk_ratio": bounded_point["ratio_of_standardized_RRs"],
        "risk_difference": bounded_point["additive_interaction_RD_difference"],
    },
])

display(clinical_scale_effects)
clinical_scale_effects.to_csv(
    TAB / "sensitivity_clinical_scale_effects.csv",
    index=False
)

,contrast,risk_ratio,risk_difference
0,CRP doubling at observed temperature,1.049015,0.027743
1,CRP doubling after temperature +1°C,1.039010,0.019648
2,Ratio of CRP-doubling risk ratios after +1°C,0.990462,-0.008095


### 22D. Missing-data analysis: neutrophil exclusion and complete-case comparison

Neutrophil percentage accounts for most of the missingness in the adjustment set and is correlated
with WBC count. The adjusted model is therefore repeated without neutrophil percentage.

Children included in the 606-child complete-case analysis are also compared with children excluded
from it, using variables that are available broadly enough for a meaningful comparison.

In [38]:
# Adjusted model without neutrophil percentage
ADJUSTMENT_VARS_NO_NEUT = [
    v for v in ADJUSTMENT_VARS if v != "Neutrophil_Percentage"
]
ADJUSTMENT_TERMS_NO_NEUT = [
    x for x in ADJUSTMENT_TERMS if x != "Neutrophil_Percentage"
]
adjusted_required_no_neut = (
    [TARGET, "z_log_CRP1", "z_Body_Temperature", "z_Body_Temperature_sq"]
    + ADJUSTMENT_VARS_NO_NEUT
)
adjusted_no_neut_data = df[adjusted_required_no_neut].dropna().copy()
ADJUSTED_FORMULA_NO_NEUT = (
    PRIMARY_FORMULA + " + " + " + ".join(ADJUSTMENT_TERMS_NO_NEUT)
)

adjusted_no_neut = smf.glm(
    ADJUSTED_FORMULA_NO_NEUT,
    data=adjusted_no_neut_data,
    family=sm.families.Poisson()
).fit(cov_type="HC0")

ci = adjusted_no_neut.conf_int().loc[term]
no_neut_summary = pd.DataFrame([{
    "analysis": "Adjusted model without neutrophil percentage",
    "n": int(adjusted_no_neut.nobs),
    "ratio_of_RRs": float(np.exp(adjusted_no_neut.params[term])),
    "CI_lower": float(np.exp(ci.iloc[0])),
    "CI_upper": float(np.exp(ci.iloc[1])),
    "p_value": float(adjusted_no_neut.pvalues[term]),
}])
display(no_neut_summary)
no_neut_summary.to_csv(
    TAB / "sensitivity_adjusted_without_neutrophils.csv",
    index=False
)

# Compare children included vs excluded from the 606 complete-case sample.
in_606 = df.index.isin(adjusted_data.index)
sample_compare = df.copy()
sample_compare["included_in_606"] = np.where(in_606, "Included", "Excluded")

compare_cont = [
    "Age", "BMI", "Body_Temperature", "WBC_Count",
    "Neutrophil_Percentage", "CRP"
]
compare_binary = [
    TARGET, "Sex", "Migratory_Pain", "Lower_Right_Abd_Pain",
    "Nausea", "Loss_of_Appetite"
]

compare_rows = []
for v in compare_cont:
    for grp, g in sample_compare.groupby("included_in_606"):
        s = pd.to_numeric(g[v], errors="coerce").dropna()
        compare_rows.append({
            "variable": v,
            "group": grp,
            "n_nonmissing": len(s),
            "summary": f"{s.mean():.3f} ± {s.std(ddof=1):.3f}",
        })

for v in compare_binary:
    for grp, g in sample_compare.groupby("included_in_606"):
        nonmiss = g[v].dropna()
        if v == TARGET:
            prop = pd.to_numeric(nonmiss, errors="coerce").mean()
            summary = f"{int(pd.to_numeric(nonmiss, errors='coerce').sum())}/{len(nonmiss)} ({100*prop:.1f}%)"
        else:
            # Report most common positive-style level as stored, without silently recoding labels.
            vc = nonmiss.astype(str).value_counts()
            summary = "; ".join([f"{k}: {int(n)}" for k,n in vc.items()])
        compare_rows.append({
            "variable": v,
            "group": grp,
            "n_nonmissing": len(nonmiss),
            "summary": summary,
        })

complete_case_comparison = pd.DataFrame(compare_rows)
display(complete_case_comparison)
complete_case_comparison.to_csv(
    TAB / "sensitivity_included_vs_excluded_606.csv",
    index=False
)

,analysis,n,ratio_of_RRs,CI_lower,CI_upper,p_value
0,Adjusted model without neutrophil percentage,692,0.907086,0.848515,0.9697,0.004191


,variable,group,n_nonmissing,summary
0,Age,Excluded,174,10.960 ± 3.874
1,Age,Included,606,11.452 ± 3.420
2,BMI,Excluded,148,18.780 ± 4.344
3,BMI,Included,606,18.936 ± 4.402
4,Body_Temperature,Excluded,168,37.535 ± 0.874
5,Body_Temperature,Included,606,37.386 ± 0.804
6,WBC_Count,Excluded,170,12.911 ± 5.748
7,WBC_Count,Included,606,12.603 ± 5.258
8,Neutrophil_Percentage,Excluded,73,76.564 ± 13.485
9,Neutrophil_Percentage,Included,606,71.216 ± 14.482


### 22E. Interaction-aware multiple imputation with passive recalculation

The outcome is included in the imputation model but is never imputed. The CRP and temperature
interaction is handled by **passive recalculation**: after each variable update, standardized CRP,
standardized temperature, squared temperature, and their product are recalculated from the current
completed data. These derived terms are then available as predictors when other variables are imputed.

This custom FCS sensitivity analysis is designed to preserve the interaction structure more closely than
imputing the component variables without updating their derived terms. It is not presented as a formal
substantive-model-compatible FCS implementation. Twenty imputations are created after 20 FCS
iterations. Convergence is checked by tracking means across iterations. Pooled coefficient inference uses
Rubin's rules with finite-m t degrees of freedom.

In [39]:
MI2_M = 20
MI2_ITER = 20
MI2_PMM_K = 5

def _derived_interaction_terms(frame):
    out = frame.copy()
    crp = pd.to_numeric(out["log_CRP1"], errors="coerce")
    temp = pd.to_numeric(out["Body_Temperature"], errors="coerce")

    zc = (crp - crp.mean()) / crp.std(ddof=1)
    zt = (temp - temp.mean()) / temp.std(ddof=1)

    out["_imp_z_log_CRP1"] = zc
    out["_imp_z_temp"] = zt
    out["_imp_z_temp_sq"] = zt ** 2
    out["_imp_product"] = zc * zt
    return out

def _design_matrix_interaction_compatible(frame, exclude_var):
    current = _derived_interaction_terms(frame)

    base_cols = [
        c for c in [TARGET] + mi_vars
        if c != exclude_var
    ]
    derived_cols = [
        "_imp_z_log_CRP1", "_imp_z_temp",
        "_imp_z_temp_sq", "_imp_product"
    ]

    # Avoid using a deterministic duplicate of the variable currently being imputed
    # when the derived term would directly contain that variable.
    if exclude_var in ["log_CRP1", "Body_Temperature"]:
        derived_cols = []

    pred = current[base_cols + derived_cols].copy()
    pred = pd.get_dummies(
        pred,
        columns=[c for c in mi_binary + mi_multiclass if c != exclude_var],
        drop_first=False,
        dtype=float
    )
    return pred.astype(float)

def _pmm_impute_ic(current, original, var, rng, k=5):
    miss = original[var].isna()
    if not miss.any():
        return
    obs = ~miss
    X = _design_matrix_interaction_compatible(current, var)
    donor_vals = pd.to_numeric(original.loc[obs, var], errors="coerce").to_numpy()

    obs_idx = np.flatnonzero(obs.to_numpy())
    boot_idx = rng.choice(obs_idx, size=len(obs_idx), replace=True)

    model = BayesianRidge()
    model.fit(
        X.iloc[boot_idx],
        pd.to_numeric(current.iloc[boot_idx][var], errors="coerce")
    )

    pred_obs = model.predict(X.loc[obs])
    pred_miss, pred_sd = model.predict(X.loc[miss], return_std=True)
    pred_miss = rng.normal(pred_miss, np.maximum(pred_sd, 1e-8))

    for row_index, pm in zip(current.index[miss], pred_miss):
        d = np.abs(pred_obs - pm)
        kk = min(k, len(d))
        donor_positions = np.argpartition(d, kk - 1)[:kk]
        chosen = rng.choice(donor_positions)
        current.at[row_index, var] = donor_vals[chosen]

def _binary_impute_ic(current, original, var, rng):
    miss = original[var].isna()
    if not miss.any():
        return
    obs = ~miss
    X = _design_matrix_interaction_compatible(current, var)
    levels = binary_levels[var]
    y_map = {levels[0]: 0, levels[1]: 1}

    obs_idx = np.flatnonzero(obs.to_numpy())
    boot_idx = rng.choice(obs_idx, size=len(obs_idx), replace=True)
    y_boot = original.iloc[boot_idx][var].astype(str).map(y_map).astype(int)

    fit_idx = boot_idx if y_boot.nunique() == 2 else obs_idx
    y_fit = original.iloc[fit_idx][var].astype(str).map(y_map).astype(int)

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, C=1e6, solver="lbfgs")
    )
    model.fit(X.iloc[fit_idx], y_fit)
    p = model.predict_proba(X.loc[miss])[:, 1]
    draw = rng.binomial(1, p)
    current.loc[miss, var] = np.where(draw == 1, levels[1], levels[0])

def _multiclass_impute_ic(current, original, var, rng):
    miss = original[var].isna()
    if not miss.any():
        return
    obs = ~miss
    X = _design_matrix_interaction_compatible(current, var)
    levels = multi_levels[var]
    level_to_int = {lev: i for i, lev in enumerate(levels)}

    obs_idx = np.flatnonzero(obs.to_numpy())
    boot_idx = rng.choice(obs_idx, size=len(obs_idx), replace=True)
    y_boot = original.iloc[boot_idx][var].astype(str).map(level_to_int).astype(int)
    fit_idx = boot_idx if y_boot.nunique() == len(levels) else obs_idx
    y_fit = original.iloc[fit_idx][var].astype(str).map(level_to_int).astype(int)

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000, C=1e6, solver="lbfgs")
    )
    model.fit(X.iloc[fit_idx], y_fit)
    probs = model.predict_proba(X.loc[miss])
    classes = model.named_steps["logisticregression"].classes_

    sampled = [
        rng.choice(classes, p=pr / pr.sum())
        for pr in probs
    ]
    inv = {i: lev for lev, i in level_to_int.items()}
    current.loc[miss, var] = [inv[int(x)] for x in sampled]

def make_imputed_dataset_ic(seed, return_trace=False):
    rng = np.random.default_rng(seed)
    current = _initial_fill(mi_base)
    trace = []

    for cycle in range(MI2_ITER):
        order = list(mi_cont + mi_binary + mi_multiclass)
        rng.shuffle(order)

        for v in order:
            if v in mi_cont:
                _pmm_impute_ic(current, mi_base, v, rng, k=MI2_PMM_K)
            elif v in mi_binary:
                _binary_impute_ic(current, mi_base, v, rng)
            else:
                _multiclass_impute_ic(current, mi_base, v, rng)

        trace.append({
            "cycle": cycle + 1,
            "mean_log_CRP1": float(pd.to_numeric(current["log_CRP1"], errors="coerce").mean()),
            "mean_temperature": float(pd.to_numeric(current["Body_Temperature"], errors="coerce").mean()),
            "mean_neutrophils": float(pd.to_numeric(current["Neutrophil_Percentage"], errors="coerce").mean()),
        })

    current["CRP"] = np.expm1(
        pd.to_numeric(current["log_CRP1"], errors="coerce")
    ).clip(lower=0)

    for v in ["log_CRP1", "Body_Temperature"]:
        current[f"z_{v}"] = (
            current[v].astype(float) - current[v].astype(float).mean()
        ) / current[v].astype(float).std(ddof=1)

    current["z_Body_Temperature_sq"] = current["z_Body_Temperature"] ** 2
    current[TARGET] = mi_base[TARGET].astype(int).values

    if return_trace:
        return current, pd.DataFrame(trace)
    return current

mi2_poisson = []
mi2_logistic = []
mi2_traces = []

for m in range(MI2_M):
    imp_df, trace_df = make_imputed_dataset_ic(
        RANDOM_STATE + 3000 + m,
        return_trace=True
    )
    trace_df["imputation"] = m + 1
    mi2_traces.append(trace_df)

    rp = smf.glm(
        ADJUSTED_FORMULA,
        data=imp_df,
        family=sm.families.Poisson()
    ).fit(cov_type="HC0")

    rl = smf.glm(
        ADJUSTED_FORMULA,
        data=imp_df,
        family=sm.families.Binomial()
    ).fit(cov_type="HC0")

    mi2_poisson.append({
        "imputation": m + 1,
        "beta": float(rp.params[term]),
        "variance": float(rp.cov_params().loc[term, term])
    })
    mi2_logistic.append({
        "imputation": m + 1,
        "beta": float(rl.params[term]),
        "variance": float(rl.cov_params().loc[term, term])
    })

mi2_poisson_pooled, mi2_poisson_components = rubin_pool(
    mi2_poisson,
    "Modified Poisson, interaction-aware MI"
)

mi2_logistic_pooled, mi2_logistic_components = rubin_pool(
    mi2_logistic,
    "Logistic regression, interaction-aware MI"
)

mi2_pooled = pd.concat(
    [mi2_poisson_pooled, mi2_logistic_pooled],
    ignore_index=True
)

mi2_trace = pd.concat(mi2_traces, ignore_index=True)

display(mi2_pooled)
display(
    mi2_trace.groupby("cycle")[
        ["mean_log_CRP1","mean_temperature","mean_neutrophils"]
    ].agg(["mean","std"]).tail(10)
)

mi2_pooled.to_csv(
    TAB / "sensitivity_interaction_compatible_MI_pooled.csv",
    index=False
)
mi2_trace.to_csv(
    TAB / "sensitivity_interaction_compatible_MI_convergence.csv",
    index=False
)

,analysis,m,pooled_exp_beta,CI_lower,CI_upper,test_statistic,df_rubin,p_value,within_imputation_variance,between_imputation_variance,relative_increase_variance,total_variance
0,"Modified Poisson, interaction-aware MI",20,0.911465,0.858463,0.967739,-3.033066,13091.920542,0.002426,0.000899,0.000034,0.039604,0.000934
1,"Logistic regression, interaction-aware MI",20,0.757768,0.602439,0.953146,-2.370129,19029.564359,0.017792,0.013263,0.000412,0.032629,0.013696


mean_log_CRP1           mean_temperature           mean_neutrophils  \
               mean       std             mean       std             mean   
cycle                                                                       
11         2.168637  0.007005        37.415353  0.001914        71.403122   
12         2.169947  0.008795        37.415455  0.001628        71.404147   
13         2.170146  0.008280        37.415750  0.003220        71.357846   
14         2.168223  0.007613        37.415654  0.002000        71.495391   
15         2.164446  0.007203        37.416224  0.002085        71.421538   
16         2.166459  0.007278        37.415564  0.002694        71.236276   
17         2.167898  0.009005        37.415462  0.002515        71.346660   
18         2.170195  0.005558        37.415199  0.001939        71.353872   
19         2.165205  0.009638        37.416474  0.002097        71.303769   
20         2.167301  0.008075        37.415994  0.002452        71.395647   

                 
            std  
cycle            
11     0.253289  
12     0.227572  
13     0.159699  
14     0.239909  
15     0.245151  
16     0.313642  
17     0.168876  
18     0.271618  
19     0.230556  
20     0.242210

### 22F. Flexible main effects and two-term temperature interaction

A large joint spline surface uses many degrees of freedom. To test the original hypothesis more directly,
a targeted logistic model with separate 4-df splines for CRP and temperature is compared with the same
model plus one linear standardized CRP$\times$temperature product term.

The previously used 2-df modified-Poisson interaction test is also reported again so it can be compared
directly with the temperature interactions in the pairwise screen.

In [40]:
targeted_df = df[[TARGET, "log_CRP1", "Body_Temperature"]].dropna().copy()
targeted_df["z_log_CRP1"] = (
    targeted_df["log_CRP1"] - targeted_df["log_CRP1"].mean()
) / targeted_df["log_CRP1"].std(ddof=1)
targeted_df["z_Body_Temperature"] = (
    targeted_df["Body_Temperature"] - targeted_df["Body_Temperature"].mean()
) / targeted_df["Body_Temperature"].std(ddof=1)

targeted_additive = smf.glm(
    f"{TARGET} ~ cr(log_CRP1, df=4) + cr(Body_Temperature, df=4)",
    data=targeted_df,
    family=sm.families.Binomial()
).fit()

targeted_product = smf.glm(
    f"{TARGET} ~ cr(log_CRP1, df=4) + cr(Body_Temperature, df=4) "
    "+ z_log_CRP1:z_Body_Temperature",
    data=targeted_df,
    family=sm.families.Binomial()
).fit()

lr_targeted = 2 * (targeted_product.llf - targeted_additive.llf)
df_targeted = int(round(targeted_product.df_model - targeted_additive.df_model))
p_targeted = chi2.sf(lr_targeted, df_targeted)

target_term = "z_log_CRP1:z_Body_Temperature"
target_ci = targeted_product.conf_int().loc[target_term]

targeted_spline_product = pd.DataFrame([{
    "n": len(targeted_df),
    "LR_chi2": lr_targeted,
    "df": df_targeted,
    "p_value": p_targeted,
    "interaction_OR": float(np.exp(targeted_product.params[target_term])),
    "CI_lower": float(np.exp(target_ci.iloc[0])),
    "CI_upper": float(np.exp(target_ci.iloc[1])),
    "AIC_additive_splines": targeted_additive.aic,
    "AIC_splines_plus_linear_product": targeted_product.aic,
}])

# Restore the adjusted two-degree-of-freedom modified-Poisson interaction test.
linear_term = "z_log_CRP1:z_Body_Temperature"
quad_term = "z_log_CRP1:z_Body_Temperature_sq"
ci_lin = poisson_interaction_nonlinear.conf_int().loc[linear_term]
ci_quad = poisson_interaction_nonlinear.conf_int().loc[quad_term]

interaction_2df_summary = pd.DataFrame([{
    "n": int(poisson_interaction_nonlinear.nobs),
    "linear_ratio_of_RRs": float(np.exp(poisson_interaction_nonlinear.params[linear_term])),
    "linear_CI_lower": float(np.exp(ci_lin.iloc[0])),
    "linear_CI_upper": float(np.exp(ci_lin.iloc[1])),
    "linear_p": float(poisson_interaction_nonlinear.pvalues[linear_term]),
    "quadratic_ratio": float(np.exp(poisson_interaction_nonlinear.params[quad_term])),
    "quadratic_CI_lower": float(np.exp(ci_quad.iloc[0])),
    "quadratic_CI_upper": float(np.exp(ci_quad.iloc[1])),
    "quadratic_p": float(poisson_interaction_nonlinear.pvalues[quad_term]),
    "joint_Wald_chi2": float(wald.statistic),
    "joint_df": 2,
    "joint_p": float(wald.pvalue),
}])

display(targeted_spline_product)
display(interaction_2df_summary)

targeted_spline_product.to_csv(
    TAB / "sensitivity_targeted_splines_plus_product.csv",
    index=False
)
interaction_2df_summary.to_csv(
    TAB / "sensitivity_restored_2df_interaction.csv",
    index=False
)

,n,LR_chi2,df,p_value,interaction_OR,CI_lower,CI_upper,AIC_additive_splines,AIC_splines_plus_linear_product
0,766,3.047947,1,0.080839,0.801766,0.623722,1.030634,904.881601,903.833654


,n,linear_ratio_of_RRs,linear_CI_lower,linear_CI_upper,linear_p,quadratic_ratio,quadratic_CI_lower,quadratic_CI_upper,quadratic_p,joint_Wald_chi2,joint_df,joint_p
0,606,0.897901,0.825941,0.976131,0.011511,1.050799,0.990613,1.114642,0.099651,7.129091,2,0.02831


### 22G. Bootstrap differences in CRP AUROC across temperature groups

The earlier logistic slope test does not directly test equality of AUROC. Here the pairwise differences
in CRP AUROC between temperature groups are estimated by bootstrap resampling within each
temperature group. Wide intervals should be interpreted as imprecision rather than evidence that the
AUROCs are equal.

In [41]:
from itertools import combinations as _combinations

AUC_DIFF_BOOT = 2000
rng = np.random.default_rng(RANDOM_STATE + 2202)

groups = {
    str(level): g.copy()
    for level, g in dx.groupby("temp_stratum", observed=True)
}

auc_diff_rows = []

for g1, g2 in _combinations(groups.keys(), 2):
    d1 = groups[g1]
    d2 = groups[g2]

    auc1 = roc_auc_score(d1[TARGET], d1["CRP"])
    auc2 = roc_auc_score(d2[TARGET], d2["CRP"])
    observed_diff = auc1 - auc2

    boot_diff = []
    for b in range(AUC_DIFF_BOOT):
        s1 = d1.iloc[rng.integers(0, len(d1), len(d1))]
        s2 = d2.iloc[rng.integers(0, len(d2), len(d2))]

        if s1[TARGET].nunique() < 2 or s2[TARGET].nunique() < 2:
            continue

        boot_diff.append(
            roc_auc_score(s1[TARGET], s1["CRP"])
            - roc_auc_score(s2[TARGET], s2["CRP"])
        )

    auc_diff_rows.append({
        "group_1": g1,
        "group_2": g2,
        "AUROC_1": auc1,
        "AUROC_2": auc2,
        "difference_AUROC_1_minus_2": observed_diff,
        "CI_lower": np.quantile(boot_diff, 0.025),
        "CI_upper": np.quantile(boot_diff, 0.975),
        "bootstrap_reps_successful": len(boot_diff),
    })

auc_difference_bootstrap = pd.DataFrame(auc_diff_rows)
display(auc_difference_bootstrap)
display(diagnostic_by_temp)

auc_difference_bootstrap.to_csv(
    TAB / "sensitivity_bootstrap_AUROC_differences.csv",
    index=False
)

# Calibration intercept AND slope were already calculated in Section 18J;
# Export a compact prediction and calibration summary.
calibration_compact = (
    repeated_calibration
    .groupby("model")[["calibration_intercept","calibration_slope"]]
    .agg(["mean","std"])
)
display(calibration_compact)
calibration_compact.to_csv(
    TAB / "sensitivity_prediction_calibration_summary.csv"
)

,group_1,group_2,AUROC_1,AUROC_2,difference_AUROC_1_minus_2,CI_lower,CI_upper,bootstrap_reps_successful
0,<37.0°C,37.0–37.9°C,0.717763,0.747442,-0.029679,-0.113720,0.054725,2000
1,<37.0°C,≥38.0°C,0.717763,0.670889,0.046874,-0.055817,0.153241,2000
2,37.0–37.9°C,≥38.0°C,0.747442,0.670889,0.076554,-0.020603,0.171500,2000


,temperature_stratum,CRP_threshold_mg_L,n,sensitivity,specificity,LR_positive,LR_negative
0,<37.0°C,10,235,0.362069,0.873950,2.872414,0.729940
1,<37.0°C,50,235,0.120690,0.983193,7.181034,0.894341
2,37.0–37.9°C,10,343,0.570048,0.757353,2.349290,0.567703
3,37.0–37.9°C,50,343,0.217391,0.955882,4.927536,0.818729
4,≥38.0°C,10,188,0.823077,0.379310,1.326068,0.466434
5,≥38.0°C,50,188,0.446154,0.706897,1.522172,0.783490


calibration_intercept           calibration_slope  \
                                     mean       std              mean   
model                                                                   
With interaction                 0.019055  0.011565          0.828537   
Without interaction              0.018511  0.011247          0.834290   

                               
                          std  
model                          
With interaction     0.025044  
Without interaction  0.023865

### 22H. Broader interaction screen: reliable-test BH correction and marker benchmarks

The primary CRP$\times$temperature question is treated as an exploratory main analysis of interest.
The 253-pair screen is a separate contextual analysis rather than the multiplicity correction for the
main interaction.

The original screen kept all 253 planned tests in the BH denominator by assigning $p=1$ to unreliable
tests. This is conservative. Here BH-FDR is also recalculated using only reliable tests.

CRP$\times$temperature is then compared with CRP$\times$WBC and CRP$\times$neutrophils, together
with CRP$\times$age and CRP$\times$hemoglobin as benchmark interactions that are less directly tied
to the same inflammatory process.

In [42]:
# ============================================================
# 22H. Broader interaction screen:
# reliable-test BH correction and marker benchmarks
# ============================================================

# ------------------------------------------------------------
# 1. BH-FDR among reliable pair-only tests only
# ------------------------------------------------------------

pair_reliable_only = pair_results.loc[
    pair_results["reliable"]
].copy()

pair_reliable_only["q_reliable_only"] = multipletests(
    pair_reliable_only["p_value"].astype(float),
    method="fdr_bh"
)[1]

# Find the CRP x temperature row regardless of ordering
mask_ct = (
    (
        (pair_reliable_only["var1"] == "CRP")
        & (pair_reliable_only["var2"] == "Body_Temperature")
    )
    |
    (
        (pair_reliable_only["var2"] == "CRP")
        & (pair_reliable_only["var1"] == "Body_Temperature")
    )
)

crp_temp_reliable_bh = pair_reliable_only.loc[
    mask_ct
].copy()

print("Reliable pair-only tests:", len(pair_reliable_only))

display(
    crp_temp_reliable_bh[
        [
            "var1",
            "var2",
            "n",
            "df_interaction",
            "p_value",
            "q_value",
            "q_reliable_only",
        ]
    ]
)

pair_reliable_only.to_csv(
    TAB / "sensitivity_pair_only_BH_reliable_tests_only.csv",
    index=False
)


# ------------------------------------------------------------
# 2. Marker and benchmark interaction comparison
#    using one common sample
# ------------------------------------------------------------

# Recover all original variables for the same children used
# in the adjusted complete-case analysis.
benchmark = df.loc[
    adjusted_data.index
].copy()

# Variables needed for every benchmark comparison
benchmark_required = [
    TARGET,
    "CRP",
    "log_CRP1",
    "Body_Temperature",
    "WBC_Count",
    "Neutrophil_Percentage",
    "Age",
    "Hemoglobin",
]

# Add all variables used in the adjustment set.
# For categorical formula terms, use their underlying raw columns.
adjustment_raw_vars = [
    "Age",
    "BMI",
    "Sex",
    "WBC_Count",
    "Neutrophil_Percentage",
    "Migratory_Pain",
    "Lower_Right_Abd_Pain",
    "Contralateral_Rebound_Tenderness",
    "Coughing_Pain",
    "Psoas_Sign",
    "Nausea",
    "Loss_of_Appetite",
    "Dysuria",
    "Peritonitis",
]

benchmark_required = list(
    dict.fromkeys(
        benchmark_required + adjustment_raw_vars
    )
)

# Use exactly the same children for all five benchmark models.
benchmark = benchmark.dropna(
    subset=benchmark_required
).copy()

print(
    "Original adjusted sample:",
    len(adjusted_data)
)

print(
    "Common benchmark sample:",
    len(benchmark)
)


# ------------------------------------------------------------
# 3. Recalculate standardized variables within this sample
# ------------------------------------------------------------

benchmark["z_log_CRP1"] = (
    benchmark["log_CRP1"]
    - benchmark["log_CRP1"].mean()
) / benchmark["log_CRP1"].std(ddof=1)

benchmark["z_Body_Temperature"] = (
    benchmark["Body_Temperature"]
    - benchmark["Body_Temperature"].mean()
) / benchmark["Body_Temperature"].std(ddof=1)

benchmark["z_Body_Temperature_sq"] = (
    benchmark["z_Body_Temperature"] ** 2
)

benchmark_vars = [
    "WBC_Count",
    "Neutrophil_Percentage",
    "Age",
    "Hemoglobin",
]

for v in benchmark_vars:
    benchmark[f"z_{v}"] = (
        benchmark[v]
        - benchmark[v].mean()
    ) / benchmark[v].std(ddof=1)


# ------------------------------------------------------------
# 4. Formula builder
# ------------------------------------------------------------

def _interaction_formula_for_benchmark(
    marker_name,
    z_name
):
    """
    Build an adjusted benchmark interaction model.

    If the marker is already present in ADJUSTMENT_TERMS,
    remove its raw main-effect term and replace it with
    the standardized version used in the interaction.
    """

    remaining = [
        x for x in ADJUSTMENT_TERMS
        if x != marker_name
    ]

    rhs = [
        "z_log_CRP1",
        z_name,
        f"z_log_CRP1:{z_name}",
        "z_Body_Temperature",
        "z_Body_Temperature_sq",
    ] + remaining

    return (
        f"{TARGET} ~ "
        + " + ".join(rhs)
    )


# ------------------------------------------------------------
# 5. Model specifications
# ------------------------------------------------------------

# Rebuild the CRP x temperature model on the SAME common
# benchmark sample, rather than using ADJUSTED_FORMULA directly.
benchmark_temp_formula = (
    f"{TARGET} ~ "
    "z_log_CRP1 + "
    "z_Body_Temperature + "
    "z_Body_Temperature_sq + "
    "z_log_CRP1:z_Body_Temperature + "
    + " + ".join(ADJUSTMENT_TERMS)
)

benchmark_specs = {

    "CRP x temperature": {
        "formula": benchmark_temp_formula,
        "term": "z_log_CRP1:z_Body_Temperature",
    },

    "CRP x WBC": {
        "formula": _interaction_formula_for_benchmark(
            "WBC_Count",
            "z_WBC_Count"
        ),
        "term": "z_log_CRP1:z_WBC_Count",
    },

    "CRP x neutrophils": {
        "formula": _interaction_formula_for_benchmark(
            "Neutrophil_Percentage",
            "z_Neutrophil_Percentage"
        ),
        "term": "z_log_CRP1:z_Neutrophil_Percentage",
    },

    "CRP x age": {
        "formula": _interaction_formula_for_benchmark(
            "Age",
            "z_Age"
        ),
        "term": "z_log_CRP1:z_Age",
    },

    "CRP x hemoglobin": {
        "formula": _interaction_formula_for_benchmark(
            "Hemoglobin",
            "z_Hemoglobin"
        ),
        "term": "z_log_CRP1:z_Hemoglobin",
    },
}


# ------------------------------------------------------------
# 6. Fit all benchmark models
# ------------------------------------------------------------

benchmark_rows = []

for label, spec in benchmark_specs.items():

    form = spec["formula"]
    interaction_term = spec["term"]

    try:

        fit = smf.glm(
            form,
            data=benchmark,
            family=sm.families.Poisson()
        ).fit(
            cov_type="HC0"
        )

        if interaction_term not in fit.params.index:
            print(
                f"WARNING: interaction term "
                f"{interaction_term} not found "
                f"for {label}; skipping."
            )
            continue

        ci = fit.conf_int().loc[
            interaction_term
        ]

        pred = np.asarray(
            fit.predict(benchmark)
        )

        benchmark_rows.append({
            "interaction": label,
            "n": int(fit.nobs),

            "ratio_of_RRs": float(
                np.exp(
                    fit.params[
                        interaction_term
                    ]
                )
            ),

            "CI_lower": float(
                np.exp(ci.iloc[0])
            ),

            "CI_upper": float(
                np.exp(ci.iloc[1])
            ),

            "p_value": float(
                fit.pvalues[
                    interaction_term
                ]
            ),

            "n_fitted_risk_gt_1": int(
                (pred > 1).sum()
            ),

            "max_fitted_risk": float(
                pred.max()
            ),
        })

    except Exception as e:

        print(
            f"FAILED: {label}\n"
            f"{type(e).__name__}: {e}"
        )


interaction_benchmark_table = pd.DataFrame(
    benchmark_rows
)

display(
    interaction_benchmark_table
)


# ------------------------------------------------------------
# 7. Save benchmark table
# ------------------------------------------------------------

interaction_benchmark_table.to_csv(
    TAB / "sensitivity_interaction_benchmarks.csv",
    index=False
)

Reliable pair-only tests: 183


,var1,var2,n,df_interaction,p_value,q_value,q_reliable_only
47,Body_Temperature,CRP,766,2.0,0.037895,0.394761,0.285538


Original adjusted sample: 606
Common benchmark sample: 603


,interaction,n,ratio_of_RRs,CI_lower,CI_upper,p_value,n_fitted_risk_gt_1,max_fitted_risk
0,CRP x temperature,603,0.921908,0.858323,0.990204,0.025751,57,1.569484
1,CRP x WBC,603,0.914445,0.861998,0.970082,0.002998,58,1.533496
2,CRP x neutrophils,603,0.869692,0.798859,0.946805,0.001277,50,1.529473
3,CRP x age,603,1.014848,0.954461,1.079056,0.637712,53,1.633174
4,CRP x hemoglobin,603,0.967588,0.919071,1.018666,0.209357,52,1.778545


### 22I. Two-part CRP analysis for recorded zeros / very low values

Because 21.7\% of non-missing CRP measurements are recorded as 0 mg/L and the smallest positive
value is 1 mg/L, the transformation constant can affect the test statistic. A two-part representation
avoids adding an arbitrary constant:

1. a binary term indicates CRP $\leq1$ mg/L versus CRP $>1$ mg/L;
2. among CRP values above 1 mg/L, $\log_2(\mathrm{CRP})$ represents a CRP doubling.

The temperature interaction is tested separately with the low-CRP indicator and with the continuous
log$_2$ CRP term.

In [43]:
two_part = df[
    [TARGET, "CRP", "Body_Temperature"] + ADJUSTMENT_VARS
].dropna().copy()

two_part["CRP_le_1"] = (two_part["CRP"] <= 1).astype(int)
two_part["log2_CRP_above1"] = np.where(
    two_part["CRP"] > 1,
    np.log2(two_part["CRP"]),
    0.0
)

two_part["z_Body_Temperature"] = (
    two_part["Body_Temperature"] - scaling["Body_Temperature"]["mean"]
) / scaling["Body_Temperature"]["sd"]
two_part["z_Body_Temperature_sq"] = two_part["z_Body_Temperature"] ** 2

two_part_adjustment = " + ".join(ADJUSTMENT_TERMS)
TWO_PART_FORMULA = (
    f"{TARGET} ~ CRP_le_1 + log2_CRP_above1 + "
    "z_Body_Temperature + z_Body_Temperature_sq + "
    "CRP_le_1:z_Body_Temperature + "
    "log2_CRP_above1:z_Body_Temperature + "
    f"{two_part_adjustment}"
)

two_part_logit = smf.glm(
    TWO_PART_FORMULA,
    data=two_part,
    family=sm.families.Binomial()
).fit(cov_type="HC0")

two_part_terms = [
    "CRP_le_1:z_Body_Temperature",
    "log2_CRP_above1:z_Body_Temperature",
]

two_part_rows = []
for t in two_part_terms:
    ci = two_part_logit.conf_int().loc[t]
    two_part_rows.append({
        "term": t,
        "n": int(two_part_logit.nobs),
        "interaction_OR": float(np.exp(two_part_logit.params[t])),
        "CI_lower": float(np.exp(ci.iloc[0])),
        "CI_upper": float(np.exp(ci.iloc[1])),
        "p_value": float(two_part_logit.pvalues[t]),
    })

# Joint Wald test for the two CRP-by-temperature terms.
R_two = np.zeros((2, len(two_part_logit.params)))
for i, t in enumerate(two_part_terms):
    R_two[i, two_part_logit.params.index.get_loc(t)] = 1
wald_two = two_part_logit.wald_test(R_two, scalar=True)

two_part_results = pd.DataFrame(two_part_rows)
display(two_part_results)
print(
    "Joint two-part CRP x temperature test:",
    f"chi2={float(wald_two.statistic):.3f}, "
    f"df=2, p={float(wald_two.pvalue):.6f}"
)

two_part_results.to_csv(
    TAB / "sensitivity_two_part_CRP_interaction.csv",
    index=False
)
pd.DataFrame([{
    "Wald_chi2": float(wald_two.statistic),
    "df": 2,
    "p_value": float(wald_two.pvalue),
}]).to_csv(
    TAB / "sensitivity_two_part_CRP_joint_test.csv",
    index=False
)

C:\Users\Asus\anaconda4\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


,term,n,interaction_OR,CI_lower,CI_upper,p_value
0,CRP_le_1:z_Body_Temperature,606,2.077230,0.906664,4.759078,0.083932
1,log2_CRP_above1:z_Body_Temperature,606,1.002786,0.869563,1.156420,0.969483


Joint two-part CRP x temperature test: chi2=5.019, df=2, p=0.081324


### 22J. Forest plots of interaction estimates

The forest plots emphasize point estimates and confidence intervals rather than classifying results as
"significant" or "not significant." Ratio-of-risk-ratio estimates are shown separately from logistic
odds-ratio estimates because these measures are not numerically equivalent.

In [44]:
# Ratio-of-risk-ratio forest plot
ror_rows = pd.DataFrame([
    {
        "analysis": "Primary modified Poisson",
        "estimate": float(primary_hc0["ratio_of_RRs"]),
        "lower": float(primary_hc0["CI_lower"]),
        "upper": float(primary_hc0["CI_upper"]),
    },
    {
        "analysis": "Same 606, unadjusted",
        "estimate": float(same606_hc0["ratio_of_RRs"]),
        "lower": float(same606_hc0["CI_lower"]),
        "upper": float(same606_hc0["CI_upper"]),
    },
    {
        "analysis": "Adjusted modified Poisson",
        "estimate": float(adjusted_hc0["ratio_of_RRs"]),
        "lower": float(adjusted_hc0["CI_lower"]),
        "upper": float(adjusted_hc0["CI_upper"]),
    },
    {
        "analysis": "Multiple imputation",
        "estimate": float(mi2_pooled.loc[
            mi2_pooled["analysis"].str.startswith("Modified Poisson"),
            "pooled_exp_beta"
        ].iloc[0]),
        "lower": float(mi2_pooled.loc[
            mi2_pooled["analysis"].str.startswith("Modified Poisson"),
            "CI_lower"
        ].iloc[0]),
        "upper": float(mi2_pooled.loc[
            mi2_pooled["analysis"].str.startswith("Modified Poisson"),
            "CI_upper"
        ].iloc[0]),
    },
])

# Add log(CRP+c) sensitivity estimates.
for _, r in constant_sensitivity.iterrows():
    ror_rows.loc[len(ror_rows)] = {
        "analysis": f"log(CRP+{r['constant_c']})",
        "estimate": r["ratio_of_RRs"],
        "lower": r["CI_lower"],
        "upper": r["CI_upper"],
    }

fig, ax = plt.subplots(figsize=(6.5, max(4.2, 0.42 * len(ror_rows))))
y = np.arange(len(ror_rows))
ax.errorbar(
    ror_rows["estimate"],
    y,
    xerr=[
        ror_rows["estimate"] - ror_rows["lower"],
        ror_rows["upper"] - ror_rows["estimate"],
    ],
    fmt="o",
    capsize=3
)
ax.axvline(1, linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(ror_rows["analysis"])
ax.invert_yaxis()
ax.set_xlabel("Ratio of risk ratios")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "sensitivity_forest_ratio_of_RRs.pdf", bbox_inches="tight")
fig.savefig(FIG / "sensitivity_forest_ratio_of_RRs.png", dpi=600, bbox_inches="tight")
plt.close(fig)

# Logistic OR forest plot
mi2_log = mi2_pooled.loc[
    mi2_pooled["analysis"].str.startswith("Logistic regression")
].iloc[0]

or_rows = pd.DataFrame([
    {
        "analysis": "Complete-case adjusted logistic",
        "estimate": float(logit_adj["interaction_OR"]),
        "lower": float(logit_adj["CI_lower"]),
        "upper": float(logit_adj["CI_upper"]),
    },
    {
        "analysis": "Multiple-imputation logistic",
        "estimate": float(mi2_log["pooled_exp_beta"]),
        "lower": float(mi2_log["CI_lower"]),
        "upper": float(mi2_log["CI_upper"]),
    },
])

fig, ax = plt.subplots(figsize=(6.5, 2.7))
y = np.arange(len(or_rows))
ax.errorbar(
    or_rows["estimate"],
    y,
    xerr=[
        or_rows["estimate"] - or_rows["lower"],
        or_rows["upper"] - or_rows["estimate"],
    ],
    fmt="o",
    capsize=3
)
ax.axvline(1, linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(or_rows["analysis"])
ax.invert_yaxis()
ax.set_xlabel("Interaction odds ratio")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "sensitivity_forest_logistic_ORs.pdf", bbox_inches="tight")
fig.savefig(FIG / "sensitivity_forest_logistic_ORs.png", dpi=600, bbox_inches="tight")
plt.close(fig)

ror_rows.to_csv(TAB / "sensitivity_forest_ratio_of_RRs_data.csv", index=False)
or_rows.to_csv(TAB / "sensitivity_forest_logistic_ORs_data.csv", index=False)

display(ror_rows)
display(or_rows)

,analysis,estimate,lower,upper
0,Primary modified Poisson,0.922840,0.865814,0.983621
1,"Same 606, unadjusted",0.924694,0.852671,1.002801
2,Adjusted modified Poisson,0.919463,0.853952,0.989999
3,Multiple imputation,0.911465,0.858463,0.967739
4,log(CRP+0.1),0.930657,0.856491,1.011246
5,log(CRP+0.5),0.924187,0.862516,0.990268
6,log(CRP+1.0),0.922840,0.865814,0.983621
7,log(CRP+2.0),0.922341,0.869584,0.978298
8,log(CRP+5.0),0.923066,0.875427,0.973298


,analysis,estimate,lower,upper
0,Complete-case adjusted logistic,0.821954,0.635107,1.063772
1,Multiple-imputation logistic,0.757768,0.602439,0.953146


### 22K. Ultrasound availability cross-tabulations

These tables report yes/no ultrasound performance explicitly and cross-tabulate appendix visualization,
diameter availability, and diagnosis. The appendix-status model is retained only as a sensitivity analysis
because the 6-mm diameter threshold contributes to the diagnosis definition in some conservatively
managed children.

In [45]:
# Explicit ultrasound performed counts.
us_performed_counts = (
    df["US_Performed"]
    .fillna("missing")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("US_Performed")
    .reset_index(name="n")
)
display(us_performed_counts)
us_performed_counts.to_csv(
    TAB / "sensitivity_ultrasound_performed_yes_no.csv",
    index=False
)

# Appendix visualization x diameter availability x diagnosis.
us_cross = df[[TARGET, "Diagnosis", "Appendix_on_US", "Appendix_Diameter"]].copy()
us_cross["Appendix_visualization"] = us_cross["Appendix_on_US"].fillna("missing").astype(str)
us_cross["Diameter_available"] = np.where(
    us_cross["Appendix_Diameter"].notna(),
    "available",
    "missing"
)

visualization_diameter_outcome = (
    us_cross
    .groupby(
        ["Appendix_visualization", "Diameter_available", "Diagnosis"],
        dropna=False
    )
    .size()
    .rename("n")
    .reset_index()
)

display(visualization_diameter_outcome)
visualization_diameter_outcome.to_csv(
    TAB / "sensitivity_ultrasound_visualization_diameter_outcome.csv",
    index=False
)

,US_Performed,n
0,yes,763
1,no,14
2,missing,3


,Appendix_visualization,Diameter_available,Diagnosis,n
0,missing,missing,appendicitis,3
1,no,available,appendicitis,1
2,no,available,no appendicitis,3
3,no,missing,appendicitis,81
4,no,missing,no appendicitis,188
5,yes,available,appendicitis,370
6,yes,available,no appendicitis,124
7,yes,missing,appendicitis,8
8,yes,missing,no appendicitis,2
